In [2]:
# Core libraries
import os
import sys
import json
import copy
import pickle
import warnings
import gc
from datetime import datetime
import numpy as np
import pandas as pd
from pathlib import Path
import shutil
from collections import defaultdict, Counter
import random
import time
from tqdm import tqdm

# Image processing
import cv2
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR, ReduceLROnPlateau, StepLR
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models
import timm

# Metrics and visualization
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve, auc,
    precision_recall_curve, average_precision_score
)

# Kaggle datasets
import kagglehub

# Suppress warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Set device for PyTorch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# Output directory for training results
OUTPUT_DIR = Path('training_results')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("✅ All libraries imported succeful")

/home/sufi/miniconda3/envs/organized/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
✅ All libraries imported succeful


In [3]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELL 77 — GLOBALS (run FIRST, always)                              ║
# ╚══════════════════════════════════════════════════════════════════════╝

import os, gc, torch, numpy as np, pandas as pd
from pathlib import Path

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# ── Semantic defect groups ─────────────────────────────────────────────
SEMANTIC_GROUPS = [
    'contamination', 'cut', 'deformation', 'fracture',
    'hole_void', 'minor_defect', 'scratch', 'surface_quality'
]

NUM_DEFECT_TYPES  = len(SEMANTIC_GROUPS)   # 8
NUM_PRODUCT_TYPES = 17

DEFECT_TYPE_TO_IDX = {name: idx for idx, name in enumerate(SEMANTIC_GROUPS)}
IDX_TO_DEFECT_TYPE = {idx: name for idx, name in enumerate(SEMANTIC_GROUPS)}

# ── Raw label → semantic group mapping ────────────────────────────────
DEFECT_GROUP_MAP = {
    # raw dataset labels
    'crack':               'fracture',
    'fracture':            'fracture',
    'faulty_imprint':      'fracture',
    'hole':                'hole_void',
    'void':                'hole_void',
    'pit':                 'hole_void',
    'blowhole':            'hole_void',
    'scratch':             'scratch',
    'score':               'scratch',
    'stain':               'surface_quality',
    'color':               'surface_quality',
    'rough':               'surface_quality',
    'uneven':              'surface_quality',
    'inclusion':           'surface_quality',
    'discolor':            'surface_quality',
    'pilling':             'surface_quality',
    'bent':                'deformation',
    'bent_lead':           'deformation',
    'squeeze':             'deformation',
    'deformation':         'deformation',
    'contamination':       'contamination',
    'glue':                'contamination',
    'oil':                 'contamination',
    'glue_strip':          'contamination',
    'liquid':              'contamination',
    'metal_contamination': 'contamination',
    'missing':             'minor_defect',
    'misplaced':           'minor_defect',
    'flip':                'minor_defect',
    'missing_hole':        'minor_defect',
    'thread':              'minor_defect',
    'cable_swap':          'minor_defect',
    'combined':            'minor_defect',
    'cut':                 'cut',
    # ── direct semantic group name passthroughs ────────────────────
    'hole_void':           'hole_void',
    'surface_quality':     'surface_quality',
    'minor_defect':        'minor_defect',
}

# ── Product type mapping ───────────────────────────────────────────────
PRODUCT_TYPE_TO_IDX = {}
IDX_TO_PRODUCT_TYPE = {}

print(f"✅ CELL 77 COMPLETE")
print(f"   NUM_DEFECT_TYPES  : {NUM_DEFECT_TYPES}")
print(f"   NUM_PRODUCT_TYPES : {NUM_PRODUCT_TYPES}")
print(f"   Semantic groups   : {SEMANTIC_GROUPS}")

Device: cuda
✅ CELL 77 COMPLETE
   NUM_DEFECT_TYPES  : 8
   NUM_PRODUCT_TYPES : 17
   Semantic groups   : ['contamination', 'cut', 'deformation', 'fracture', 'hole_void', 'minor_defect', 'scratch', 'surface_quality']


In [6]:
import torch
import torch.nn as nn
import torchvision.models as models

class CoordAttention(nn.Module):
    def __init__(self, channels, reduction=32):
        super().__init__()
        mid = max(8, channels // reduction)
        self.conv1  = nn.Conv2d(channels, mid, 1, bias=False)
        self.bn1    = nn.BatchNorm2d(mid)
        self.act    = nn.ReLU()
        self.conv_h = nn.Conv2d(mid, channels, 1, bias=False)
        self.conv_w = nn.Conv2d(mid, channels, 1, bias=False)

    def forward(self, x):
        B, C, H, W = x.shape
        x_h = x.mean(dim=3, keepdim=True)
        x_w = x.mean(dim=2, keepdim=True).permute(0, 1, 3, 2)
        y   = torch.cat([x_h, x_w], dim=2)
        y   = self.act(self.bn1(self.conv1(y)))
        x_h, x_w = torch.split(y, [H, W], dim=2)
        x_w = x_w.permute(0, 1, 3, 2)
        return x * torch.sigmoid(self.conv_h(x_h)) * torch.sigmoid(self.conv_w(x_w))


class EdgeNetV2(nn.Module):
    def __init__(self, num_defect=8, num_product=17):
        super().__init__()
        base = models.mobilenet_v3_large(
            weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V2)

        self.early_features = base.features[:7]
        self.coord_att1     = CoordAttention(40)
        self.late_features  = base.features[7:]
        self.coord_att2     = CoordAttention(960)
        self.pool           = nn.AdaptiveAvgPool2d(1)

        self.shared = nn.Sequential(
            nn.Linear(960, 512),   # shared.0
            nn.Hardswish(),
            nn.Dropout(0.2),
        )

        # ✅ FIXED: 512 → 128 → 1   (confirmed from checkpoint shape [128,512])
        self.binary_head = nn.Sequential(
            nn.Linear(512, 128),   # binary_head.0
            nn.Hardswish(),
            nn.Dropout(0.2),
            nn.Linear(128, 1),     # binary_head.3
        )

        # defect_head: 512 → 256 → num_defect
        self.defect_head = nn.Sequential(
            nn.Linear(512, 256),   # defect_head.0
            nn.Hardswish(),
            nn.Dropout(0.3),
            nn.Linear(256, num_defect),  # defect_head.3
        )

        # product_head: 512 → 256 → num_product
        self.product_head = nn.Sequential(
            nn.Linear(512, 256),   # product_head.0
            nn.Hardswish(),
            nn.Dropout(0.2),
            nn.Linear(256, num_product),  # product_head.3
        )

    def freeze_backbone(self, freeze=True):
        for p in self.early_features.parameters():
            p.requires_grad = not freeze
        for p in self.late_features.parameters():
            p.requires_grad = not freeze

    def count_params(self, trainable_only=False):
        if trainable_only:
            return sum(p.numel() for p in self.parameters() if p.requires_grad)
        return sum(p.numel() for p in self.parameters())

    def forward(self, x):
        x = self.early_features(x)
        x = self.coord_att1(x)
        x = self.late_features(x)
        x = self.coord_att2(x)
        x = self.pool(x).flatten(1)
        x = self.shared(x)
        return self.binary_head(x), self.defect_head(x), self.product_head(x)


# ── Build and load ────────────────────────────────────────────────────
model = EdgeNetV2(
    num_defect  = NUM_DEFECT_TYPES,
    num_product = NUM_PRODUCT_TYPES,
).to(device)

CKPT = '/home/sufi/training_results/models/EdgeNet_V2.pth'
ck   = torch.load(CKPT, map_location=device, weights_only=False)
missing, unexpected = model.load_state_dict(ck['model_state_dict'], strict=False)

print(f"Total params   : {model.count_params()/1e6:.2f}M")
print(f"Missing keys   : {len(missing)}")
print(f"Unexpected keys: {len(unexpected)}")

if len(missing) == 0 and len(unexpected) == 0:
    print("✅ PERFECT MATCH")
    model.eval()
    vm = ck['val_metrics']
    print(f"Epoch {ck['epoch']} | Binary={vm['binary_acc']:.1f}% "
          f"Defect={vm['defect_acc']:.1f}% F1={vm['defect_f1_macro']:.3f} "
          f"Product={vm['product_acc']:.1f}%")
else:
    # Auto-print shape mismatches for debugging
    s_sd = ck['model_state_dict']
    print("⚠️  Still mismatched — shape comparison:")
    for k in list(missing)[:5]:
        ck_shape = s_sd[k].shape if k in s_sd else "NOT IN CKPT"
        model_shape = dict(model.named_parameters()).get(k, None)
        model_shape = model_shape.shape if model_shape is not None else "NOT IN MODEL"
        print(f"  {k}: ckpt={ck_shape}  model={model_shape}")

print("\n✅ CELL 1 COMPLETE")

Total params   : 3.89M
Missing keys   : 0
Unexpected keys: 0
✅ PERFECT MATCH
Epoch 87 | Binary=98.1% Defect=86.8% F1=0.844 Product=99.9%

✅ CELL 1 COMPLETE


In [7]:
# ======================================================================
# ENH_CELL_1 — DEFINITIONS: FeatureProjector + SupConLoss
# ======================================================================
import torch
import torch.nn as nn
import torch.nn.functional as F

class FeatureProjector(nn.Module):
    """
    Projects student pool features (960) into teacher space (1280).
    Two-layer MLP with ReLU — avoids trivial collapse.
    """
    def __init__(self, student_dim=960, teacher_dim=1280, hidden_dim=1024):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(student_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, teacher_dim),
        )

    def forward(self, x):
        return self.proj(x)


class SupConLoss(nn.Module):
    """
    Supervised Contrastive Loss (Khosla et al., 2020).
    Pulls same-class embeddings together, pushes different-class apart.
    Applied to 512-dim shared embeddings, defect-labeled samples only.
    """
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temp = temperature

    def forward(self, features, labels):
        """
        features : [N, D]  (raw, will be L2-normalised inside)
        labels   : [N]     (LongTensor, no -1 values — filter before calling)
        """
        device  = features.device
        N       = features.shape[0]
        if N < 2:
            return torch.tensor(0.0, device=device)

        # L2-normalise
        features = F.normalize(features, dim=1)

        # Similarity matrix [N, N]
        sim = torch.matmul(features, features.T) / self.temp

        # Numerical stability
        sim = sim - sim.max(dim=1, keepdim=True).values.detach()

        # Masks
        mask_self = torch.eye(N, dtype=torch.bool, device=device)
        labels_row = labels.unsqueeze(1)               # [N,1]
        mask_pos   = (labels_row == labels_row.T) & ~mask_self  # [N,N]

        # Skip if no positive pairs at all
        if mask_pos.sum() == 0:
            return torch.tensor(0.0, device=device)

        exp_sim     = torch.exp(sim) * (~mask_self).float()
        log_prob    = sim - torch.log(exp_sim.sum(dim=1, keepdim=True) + 1e-8)

        # Mean log-prob over positives, per anchor
        n_pos       = mask_pos.float().sum(dim=1)          # [N]
        loss_per    = -(log_prob * mask_pos.float()).sum(dim=1) / (n_pos + 1e-8)

        # Average only over anchors that have ≥1 positive
        has_pos = n_pos > 0
        if has_pos.sum() == 0:
            return torch.tensor(0.0, device=device)
        return loss_per[has_pos].mean()


def cosine_feat_loss(s_feat, t_feat):
    """
    Feature alignment: maximise cosine similarity (= minimise 1 - cos_sim).
    Both tensors are L2-normalised before comparison.
    """
    s = F.normalize(s_feat, dim=1)
    t = F.normalize(t_feat, dim=1)
    return (1.0 - (s * t).sum(dim=1)).mean()


# ── Instantiate ────────────────────────────────────────────────────────
projector  = FeatureProjector(student_dim=960, teacher_dim=1280).to(device)
supcon     = SupConLoss(temperature=0.07)

print("✅ ENH_CELL_1 COMPLETE")
print(f"   FeatureProjector params : "
      f"{sum(p.numel() for p in projector.parameters())/1e6:.3f}M")
print(f"   SupConLoss temperature  : 0.07")
print(f"   Feature alignment       : cosine loss (960→1280)")

✅ ENH_CELL_1 COMPLETE
   FeatureProjector params : 2.296M
   SupConLoss temperature  : 0.07
   Feature alignment       : cosine loss (960→1280)


In [9]:
# ────────────────────────────────────────────────────────────────────────
# CELL KD1 — FIXED: Teacher matches exact checkpoint key structure
# ────────────────────────────────────────────────────────────────────────

import torch
import torch.nn as nn
import torchvision.models as models
from pathlib import Path

class EfficientNetTeacher(nn.Module):
    """
    Architecture matches exactly what was saved in efficientnet_b0_best.pth.
    Key names confirmed from checkpoint: features.*, shared.0, 
    binary_head.0/2, defect_head.0/3, product_head.0/2
    """
    def __init__(self, num_defect_classes=8, num_product_classes=17):
        super().__init__()
        base           = models.efficientnet_b0(
            weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        self.features  = base.features        # ← must be 'features' not 'backbone'
        self.pool      = nn.AdaptiveAvgPool2d(1)

        # shared.0 = single Linear
        self.shared = nn.Sequential(
            nn.Linear(1280, 512),             # shared.0
            nn.SiLU(),
            nn.Dropout(0.3),
        )

        # binary_head: index 0=Linear, 1=ReLU/dropout, 2=Linear
        self.binary_head = nn.Sequential(
            nn.Linear(512, 128),              # binary_head.0
            nn.ReLU(),
            nn.Linear(128, 1),               # binary_head.2
        )

        # defect_head: index 0=Linear, 1,2=act/drop, 3=Linear
        self.defect_head = nn.Sequential(
            nn.Linear(512, 256),              # defect_head.0
            nn.SiLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_defect_classes), # defect_head.3
        )

        # product_head: index 0=Linear, 1=act/drop, 2=Linear
        self.product_head = nn.Sequential(
            nn.Linear(512, 256),              # product_head.0
            nn.SiLU(),
            nn.Linear(256, num_product_classes), # product_head.2
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x).flatten(1)
        s = self.shared(x)
        return self.binary_head(s), self.defect_head(s), self.product_head(s)


# ── Load teacher ──────────────────────────────────────────────────────
TEACHER_CKPT = Path('/home/sufi/training_results/baseline_models/efficientnet_b0_best.pth')

teacher = EfficientNetTeacher(
    num_defect_classes  = NUM_DEFECT_TYPES,
    num_product_classes = NUM_PRODUCT_TYPES,
).to(device)

ck = torch.load(TEACHER_CKPT, map_location=device, weights_only=False)
# checkpoint IS the raw state dict — load directly
teacher.load_state_dict(ck, strict=False)

teacher.eval()
for p in teacher.parameters():
    p.requires_grad = False

params_t = sum(p.numel() for p in teacher.parameters()) / 1e6
print(f"✅ CELL KD1 COMPLETE — Teacher loaded")
print(f"   Params  : {params_t:.2f}M  (frozen)")

# ── Verify ────────────────────────────────────────────────────────────
with torch.no_grad():
    dummy    = torch.randn(2, 3, 224, 224).to(device)
    tb, td, tp = teacher(dummy)
    print(f"   Outputs : bin={tb.shape} def={td.shape} prod={tp.shape}  ✅")

✅ CELL KD1 COMPLETE — Teacher loaded
   Params  : 5.00M  (frozen)
   Outputs : bin=torch.Size([2, 1]) def=torch.Size([2, 8]) prod=torch.Size([2, 17])  ✅


In [10]:
# ======================================================================
# ENH_CELL_2 — SETUP: Load KD checkpoint, register hooks, build optimizer
# ======================================================================
from pathlib import Path

ENH_SAVE_PATH = Path('/home/sufi/training_results/models/EdgeNet_V2_ENH.pth')
ENH_EPOCHS    = 30
LAMBDA_FEAT   = 0.4    # weight for feature alignment loss
LAMBDA_CON    = 0.3    # weight for supervised contrastive loss
ALPHA_FIXED   = 0.5    # hard/soft balance — fixed (already annealed in KD)

# ── Load student from KD best checkpoint ──────────────────────────────
ck_kd = torch.load(
    '/home/sufi/training_results/models/EdgeNet_V2_KD.pth',
    map_location=device, weights_only=False)
model.load_state_dict(ck_kd['model_state_dict'])
model.freeze_backbone(False)          # all params trainable
print(f"Student loaded from KD epoch {ck_kd['epoch']} "
      f"(val F1={ck_kd['val_metrics']['defect_f1_macro']:.3f})")

# Teacher already loaded and frozen from KD_CELL_1 / KD_CELL_2
teacher.eval()
for p in teacher.parameters():
    p.requires_grad = False

# ── Register forward hooks to capture pool-level features ─────────────
# Student: AdaptiveAvgPool2d output → flatten → 960-dim
# Teacher: AdaptiveAvgPool2d output → flatten → 1280-dim
# Shared embedding: after model.shared → 512-dim (for SupCon)

_student_pool_feat   = {}
_teacher_pool_feat   = {}
_student_shared_feat = {}

def _hook_student_pool(module, inp, out):
    _student_pool_feat['feat'] = out.flatten(1)   # [B, 960]

def _hook_teacher_pool(module, inp, out):
    _teacher_pool_feat['feat'] = out.flatten(1)   # [B, 1280]

def _hook_student_shared(module, inp, out):
    _student_shared_feat['feat'] = out            # [B, 512]

h1 = model.pool.register_forward_hook(_hook_student_pool)
h2 = teacher.pool.register_forward_hook(_hook_teacher_pool)
h3 = model.shared.register_forward_hook(_hook_student_shared)

# Verify hooks work
with torch.no_grad():
    _d = torch.randn(2, 3, 224, 224).to(device)
    model(_d); teacher(_d)
    assert _student_pool_feat['feat'].shape   == (2, 960),  "student pool hook failed"
    assert _teacher_pool_feat['feat'].shape   == (2, 1280), "teacher pool hook failed"
    assert _student_shared_feat['feat'].shape == (2, 512),  "student shared hook failed"
print("✅ All hooks verified")

# ── Optimizer ─────────────────────────────────────────────────────────
# Very low LR — starting from strong KD checkpoint
bb_params    = list(model.early_features.parameters()) + \
               list(model.late_features.parameters())
coord_params = list(model.coord_att1.parameters()) + \
               list(model.coord_att2.parameters())
head_params  = list(model.shared.parameters())      + \
               list(model.binary_head.parameters()) + \
               list(model.defect_head.parameters()) + \
               list(model.product_head.parameters())
proj_params  = list(projector.parameters())

optimizer_enh = torch.optim.AdamW([
    {'params': bb_params,    'lr': 1e-5},
    {'params': coord_params, 'lr': 5e-5},
    {'params': head_params,  'lr': 5e-5},
    {'params': proj_params,  'lr': 1e-4},
], weight_decay=1e-4)

scheduler_enh = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_enh, T_max=ENH_EPOCHS, eta_min=1e-7)

from torch.cuda.amp import GradScaler
scaler_enh = GradScaler()

print(f"\n✅ ENH_CELL_2 COMPLETE")
print(f"   Starting from   : KD epoch {ck_kd['epoch']}, val F1={ck_kd['val_score']:.3f}")
print(f"   ENH epochs      : {ENH_EPOCHS}")
print(f"   λ_feat          : {LAMBDA_FEAT}")
print(f"   λ_supcon        : {LAMBDA_CON}")
print(f"   α (hard/soft)   : {ALPHA_FIXED} (fixed)")
print(f"   LR (bb/coord/heads/proj): 1e-5 / 5e-5 / 5e-5 / 1e-4")

Student loaded from KD epoch 88 (val F1=0.882)
✅ All hooks verified

✅ ENH_CELL_2 COMPLETE
   Starting from   : KD epoch 88, val F1=0.882
   ENH epochs      : 30
   λ_feat          : 0.4
   λ_supcon        : 0.3
   α (hard/soft)   : 0.5 (fixed)
   LR (bb/coord/heads/proj): 1e-5 / 5e-5 / 5e-5 / 1e-4


In [13]:
# ── AUG FIX — FIXED VERSION ───────────────────────────────────────────

CSV_PATH = '/home/sufi/merged_dataset_metadata_augmented.csv'
df_3head = pd.read_csv(CSV_PATH)

# ── Safe rename — only rename if old names exist ─────────────────────
if 'path' in df_3head.columns:
    df_3head = df_3head.rename(columns={'path': 'image_path'})
if 'category' in df_3head.columns:
    df_3head = df_3head.rename(columns={'category': 'product_type'})

print(f"Loaded CSV : {len(df_3head):,} rows")
print(f"Columns    : {list(df_3head.columns)}")

# ── Binary label ──────────────────────────────────────────────────────
def compute_binary(v):
    if pd.isna(v): return 0
    return 0 if str(v).strip().lower() in (
        '0','good','normal','ok','casting_ok') else 1

df_3head['binary_label'] = df_3head['label'].apply(compute_binary)
print(f"\n   binary_label : {df_3head['binary_label'].value_counts().to_dict()}")

# ── Defect label ──────────────────────────────────────────────────────
def remap_defect(raw):
    if pd.isna(raw): return -1
    s   = str(raw).strip().lower()
    if s in ('good','normal','casting_ok',''): return -1
    sem = DEFECT_GROUP_MAP.get(s, None)
    if sem is None and s in SEMANTIC_GROUPS: sem = s
    if sem is None:
        for k, v in DEFECT_GROUP_MAP.items():
            if k in s: sem = v; break
    return -1 if sem is None else DEFECT_TYPE_TO_IDX.get(sem, -1)

df_3head['defect_type_label'] = df_3head['defect_type'].apply(remap_defect)
df_3head.loc[df_3head['binary_label'] == 0, 'defect_type_label'] = -1

# ── Product label — use explicit loop to avoid numpy.matrix error ─────
all_products        = sorted(df_3head['product_type'].dropna().unique().tolist())
PRODUCT_TYPE_TO_IDX = {p: i for i, p in enumerate(all_products)}
IDX_TO_PRODUCT_TYPE = {i: p for i, p in enumerate(all_products)}
NUM_PRODUCT_TYPES   = len(all_products)

# Apply manually — avoids pandas .map() numpy.matrix bug
prod_labels = []
for v in df_3head['product_type']:
    if pd.isna(v):
        prod_labels.append(0)
    else:
        prod_labels.append(PRODUCT_TYPE_TO_IDX.get(str(v), 0))
df_3head['product_type_label'] = prod_labels

# ── Split ─────────────────────────────────────────────────────────────
train_df = df_3head[df_3head['split'] == 'train'].reset_index(drop=True)
val_df   = df_3head[df_3head['split'] == 'val'].reset_index(drop=True)
test_df  = df_3head[df_3head['split'] == 'test'].reset_index(drop=True)

unique = sorted(df_3head[
    df_3head['defect_type_label'] != -1
]['defect_type_label'].unique().tolist())

print(f"\n   Products       : {NUM_PRODUCT_TYPES} → {all_products}")
print(f"   Defect labels  : {unique}")
print(f"   Train          : {len(train_df):,}")
print(f"   Val            : {len(val_df):,}")
print(f"   Test           : {len(test_df):,}")

print(f"\n📊 Val samples per defect class:")
for i, name in enumerate(SEMANTIC_GROUPS):
    n = (val_df['defect_type_label'] == i).sum()
    flag = '  ⚠️ LOW' if n < 15 else ''
    print(f"   [{i}] {name:<20} val={n:>4}{flag}")

print("\n✅ AUG FIX COMPLETE")

Loaded CSV : 21,344 rows
Columns    : ['image_path', 'dataset', 'product_type', 'defect_type', 'label', 'split', 'defect_group', 'defect_type_label', 'binary_label']

   binary_label : {1: 11688, 0: 9656}

   Products       : 17 → ['bottle', 'cable', 'capsule', 'carpet', 'casting_product', 'grid', 'hazelnut', 'leather', 'magnetic_tile', 'metal_nut', 'pill', 'screw', 'tile', 'toothbrush', 'transistor', 'wood', 'zipper']
   Defect labels  : [0, 1, 2, 3, 4, 5, 6, 7]
   Train          : 16,337
   Val            : 3,339
   Test           : 1,668

📊 Val samples per defect class:
   [0] contamination        val=  33
   [1] cut                  val=  17
   [2] deformation          val=  15
   [3] fracture             val=  43
   [4] hole_void            val=  54
   [5] minor_defect         val=  42
   [6] scratch              val=  29
   [7] surface_quality      val=  63

✅ AUG FIX COMPLETE


In [14]:
# ======================================================================
# KD_CELL_3B — BUILD criterion_dict  (run before KD_CELL_4)
# ======================================================================
import torch
import torch.nn as nn
import torch.nn.functional as F

# ── Per-class focal loss ───────────────────────────────────────────────
class FocalLoss(nn.Module):
    def __init__(self, gamma_per_class, weight=None, ignore_index=-1):
        super().__init__()
        self.gamma        = torch.tensor(gamma_per_class, dtype=torch.float32)
        self.weight       = weight
        self.ignore_index = ignore_index

    def forward(self, logits, targets):
        valid   = targets != self.ignore_index
        if valid.sum() == 0:
            return logits.sum() * 0.0
        logits  = logits[valid]
        targets = targets[valid]
        gamma   = self.gamma.to(logits.device)
        ce      = F.cross_entropy(logits, targets,
                                  weight=self.weight,
                                  reduction='none')
        pt      = torch.exp(-ce)
        g       = gamma[targets]
        return ((1 - pt) ** g * ce).mean()


# ── Homoscedastic uncertainty multi-task weighting ────────────────────
class HomoscedasticLoss(nn.Module):
    """
    Learnable log-variance per task.
    loss_i = exp(-log_var_i) * task_loss_i + log_var_i
    """
    def __init__(self, n_tasks=3):
        super().__init__()
        self.log_vars = nn.Parameter(torch.zeros(n_tasks))

    def weights(self):
        return torch.exp(-self.log_vars).detach()

    def set_epoch(self, epoch):
        pass   # no-op; kept for API compatibility

    def state_dict(self, **kwargs):
        return super().state_dict(**kwargs)

    def forward(self, losses):
        w = torch.exp(-self.log_vars)
        return sum(w[i] * losses[i] + self.log_vars[i]
                   for i in range(len(losses)))


# ── Compute class weights from training set ───────────────────────────
def compute_defect_weights(train_df, num_classes, device):
    counts = torch.zeros(num_classes)
    for i in range(num_classes):
        counts[i] = max(1, (train_df['defect_type_label'] == i).sum())
    w = 1.0 / counts
    return (w / w.sum() * num_classes).to(device)

defect_weights = compute_defect_weights(train_df, NUM_DEFECT_TYPES, device)

# Gamma per defect class (thesis values)
# contamination=2.0, cut=2.0, deformation=3.5, fracture=2.0,
# hole_void=2.0, minor_defect=2.0, scratch=4.0, surface_quality=2.0
GAMMA_PER_CLASS = [2.0, 2.0, 3.5, 2.0, 2.0, 2.0, 4.0, 2.0]

# Positive-class weight for binary head (defective:normal ratio)
n_defective = (train_df['binary_label'] == 1).sum()
n_normal    = (train_df['binary_label'] == 0).sum()
pos_weight  = torch.tensor([n_normal / max(n_defective, 1)],
                            dtype=torch.float32).to(device)

criterion_dict = {
    'binary':    nn.BCEWithLogitsLoss(pos_weight=pos_weight),
    'defect':    FocalLoss(gamma_per_class=GAMMA_PER_CLASS,
                           weight=defect_weights,
                           ignore_index=-1),
    'product':   nn.CrossEntropyLoss(),
    'multitask': HomoscedasticLoss(n_tasks=3).to(device),
}

print("✅ KD_CELL_3B COMPLETE — criterion_dict built")
print(f"   Keys          : {list(criterion_dict.keys())}")
print(f"   pos_weight    : {pos_weight.item():.3f}")
print(f"   defect_weights: {[f'{w:.3f}' for w in defect_weights.cpu()]}")
print(f"   gamma/class   : {GAMMA_PER_CLASS}")
tw = criterion_dict['multitask'].weights()
print(f"   task weights  : {tw}")

✅ KD_CELL_3B COMPLETE — criterion_dict built
   Keys          : ['binary', 'defect', 'product', 'multitask']
   pos_weight    : 0.706
   defect_weights: ['0.601', '0.893', '0.459', '0.738', '1.039', '2.756', '0.512', '1.002']
   gamma/class   : [2.0, 2.0, 3.5, 2.0, 2.0, 2.0, 4.0, 2.0]
   task weights  : tensor([1., 1., 1.], device='cuda:0')


In [18]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELL 78 — DATASET                                                  ║
# ╚══════════════════════════════════════════════════════════════════════╝

import torch
from torch.utils.data import Dataset, DataLoader
import cv2
from PIL import Image
import torchvision.transforms as transforms

class ThreeHeadDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df        = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # ── load image ───────────────────────────────────────────
        img_path = row['image_path']
        img      = cv2.imread(img_path)
        if img is None:
            img = np.zeros((224, 224, 3), dtype=np.uint8)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = Image.fromarray(img)

        if self.transform:
            img = self.transform(img)

        binary_label  = int(row['binary_label'])
        defect_label  = int(row['defect_type_label'])
        product_label = int(row['product_type_label'])

        return img, binary_label, defect_label, product_label, img_path

# ── ADD THIS TO THE BOTTOM OF CELL 78 (after ThreeHeadDataset class) ──

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(0.3, 0.3, 0.2, 0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

train_3head_ds = ThreeHeadDataset(train_df, transform=train_transform)
val_3head_ds   = ThreeHeadDataset(val_df,   transform=val_transform)
test_3head_ds  = ThreeHeadDataset(test_df,  transform=val_transform)

print(f"✅ CELL 78 COMPLETE")
print(f"   train_3head_ds : {len(train_3head_ds):,} samples")
print(f"   val_3head_ds   : {len(val_3head_ds):,} samples")
print(f"   test_3head_ds  : {len(test_3head_ds):,} samples")

print("✅ CELL 78 COMPLETE — ThreeHeadDataset defined")

✅ CELL 78 COMPLETE
   train_3head_ds : 16,337 samples
   val_3head_ds   : 3,339 samples
   test_3head_ds  : 1,668 samples
✅ CELL 78 COMPLETE — ThreeHeadDataset defined


In [19]:
# ════════════════════════════════════════════════════════════════════════
# CELL 78C — CUTMIX DATALOADER (replaces Cell 78B)
# Run AFTER Cell 78 (which defines ThreeHeadDataset)
# ════════════════════════════════════════════════════════════════════════
 
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, WeightedRandomSampler
import numpy as np
from sklearn.utils.class_weight import compute_class_weight
 
# ── Weighted sampler (same as Cell 78B) ───────────────────────────────
def make_weighted_sampler(df):
    class_counts = np.ones(NUM_DEFECT_TYPES)
    for i in range(NUM_DEFECT_TYPES):
        class_counts[i] = max(1, (df['defect_type_label'] == i).sum())
    cw = 1.0 / class_counts
    cw = cw / cw.sum()
    weights = []
    for _, row in df.iterrows():
        lbl = int(row['defect_type_label'])
        weights.append(0.3 if lbl == -1 else float(cw[lbl]))
    weights = torch.tensor(weights, dtype=torch.float32)
    return WeightedRandomSampler(weights, len(weights), replacement=True)
 
sampler = make_weighted_sampler(train_df)
 
# ── CutMix collate function ────────────────────────────────────────────
def cutmix_collate(batch):
    """
    Standard collate + CutMix applied only to defect-labeled samples.
    Binary and product labels unchanged (CutMix only on defect head).
    """
    imgs, bin_l, def_l, prod_l, paths = zip(*batch)
    imgs   = torch.stack(imgs)
    bin_l  = torch.tensor(bin_l,  dtype=torch.long)
    def_l  = torch.tensor(def_l,  dtype=torch.long)
    prod_l = torch.tensor(prod_l, dtype=torch.long)
 
    # Only apply CutMix to samples with known defect labels
    known = (def_l != -1).nonzero(as_tuple=True)[0]
 
    if len(known) >= 4 and torch.rand(1).item() < 0.5:
        # Sample random lambda from Beta(1.0, 1.0)
        lam   = float(np.random.beta(1.0, 1.0))
        lam   = max(0.3, min(0.7, lam))   # constrain so neither label dominates
 
        # Shuffle known indices to get pairs
        perm  = known[torch.randperm(len(known))]
 
        B, C, H, W = imgs[known].shape
        # Random bounding box
        cut_ratio = np.sqrt(1 - lam)
        cut_h     = int(H * cut_ratio)
        cut_w     = int(W * cut_ratio)
        cx        = np.random.randint(W)
        cy        = np.random.randint(H)
        x1        = max(0, cx - cut_w // 2)
        x2        = min(W, cx + cut_w // 2)
        y1        = max(0, cy - cut_h // 2)
        y2        = min(H, cy + cut_h // 2)
 
        # Apply cut and paste
        imgs_copy = imgs.clone()
        imgs_copy[known, :, y1:y2, x1:x2] = imgs[perm, :, y1:y2, x1:x2]
 
        # Adjust lambda to actual area
        lam_actual = 1 - (x2 - x1) * (y2 - y1) / (H * W)
 
        # Soft labels for defect head only
        # Store both original and mixed labels; loss will handle soft targets
        # We use a simple approach: randomly assign label based on lambda
        # (avoids changing loss function signature)
        # For samples where lam > 0.5 keep original label, else swap
        if lam_actual >= 0.5:
            imgs = imgs_copy
        else:
            imgs = imgs_copy
            def_l_new = def_l.clone()
            def_l_new[known] = def_l[perm]
            def_l = def_l_new
 
    return imgs, bin_l, def_l, prod_l, list(paths)
 
 
# ── Build dataloaders ──────────────────────────────────────────────────
BATCH_SIZE  = 32
NUM_WORKERS = 4
 
train_3head_loader = DataLoader(
    train_3head_ds,
    batch_size  = BATCH_SIZE,
    sampler     = sampler,
    num_workers = NUM_WORKERS,
    pin_memory  = True,
    drop_last   = True,
    collate_fn  = cutmix_collate,   # ← CutMix applied here
)
 
val_3head_loader = DataLoader(
    val_3head_ds,
    batch_size  = BATCH_SIZE,
    shuffle     = False,
    num_workers = NUM_WORKERS,
    pin_memory  = True,
    # No CutMix on val — clean evaluation
)
 
print(f"✅ CELL 78C COMPLETE — CutMix DataLoader")
print(f"   Train batches : {len(train_3head_loader)}")
print(f"   Val batches   : {len(val_3head_loader)}")
print(f"   CutMix        : ✅ applied in collate (p=0.5 per batch)")
print(f"   Weighted sampler: ✅ active")

✅ CELL 78C COMPLETE — CutMix DataLoader
   Train batches : 510
   Val batches   : 105
   CutMix        : ✅ applied in collate (p=0.5 per batch)
   Weighted sampler: ✅ active


In [20]:
# ════════════════════════════════════════════════════════════════════════
# CELL 78C — CUTMIX DATALOADER (replaces Cell 78B)
# Run AFTER Cell 78 (which defines ThreeHeadDataset)
# ════════════════════════════════════════════════════════════════════════
 
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, WeightedRandomSampler
import numpy as np
from sklearn.utils.class_weight import compute_class_weight
 
# ── Weighted sampler (same as Cell 78B) ───────────────────────────────
def make_weighted_sampler(df):
    class_counts = np.ones(NUM_DEFECT_TYPES)
    for i in range(NUM_DEFECT_TYPES):
        class_counts[i] = max(1, (df['defect_type_label'] == i).sum())
    cw = 1.0 / class_counts
    cw = cw / cw.sum()
    weights = []
    for _, row in df.iterrows():
        lbl = int(row['defect_type_label'])
        weights.append(0.3 if lbl == -1 else float(cw[lbl]))
    weights = torch.tensor(weights, dtype=torch.float32)
    return WeightedRandomSampler(weights, len(weights), replacement=True)
 
sampler = make_weighted_sampler(train_df)
 
# ── CutMix collate function ────────────────────────────────────────────
def cutmix_collate(batch):
    """
    Standard collate + CutMix applied only to defect-labeled samples.
    Binary and product labels unchanged (CutMix only on defect head).
    """
    imgs, bin_l, def_l, prod_l, paths = zip(*batch)
    imgs   = torch.stack(imgs)
    bin_l  = torch.tensor(bin_l,  dtype=torch.long)
    def_l  = torch.tensor(def_l,  dtype=torch.long)
    prod_l = torch.tensor(prod_l, dtype=torch.long)
 
    # Only apply CutMix to samples with known defect labels
    known = (def_l != -1).nonzero(as_tuple=True)[0]
 
    if len(known) >= 4 and torch.rand(1).item() < 0.5:
        # Sample random lambda from Beta(1.0, 1.0)
        lam   = float(np.random.beta(1.0, 1.0))
        lam   = max(0.3, min(0.7, lam))   # constrain so neither label dominates
 
        # Shuffle known indices to get pairs
        perm  = known[torch.randperm(len(known))]
 
        B, C, H, W = imgs[known].shape
        # Random bounding box
        cut_ratio = np.sqrt(1 - lam)
        cut_h     = int(H * cut_ratio)
        cut_w     = int(W * cut_ratio)
        cx        = np.random.randint(W)
        cy        = np.random.randint(H)
        x1        = max(0, cx - cut_w // 2)
        x2        = min(W, cx + cut_w // 2)
        y1        = max(0, cy - cut_h // 2)
        y2        = min(H, cy + cut_h // 2)
 
        # Apply cut and paste
        imgs_copy = imgs.clone()
        imgs_copy[known, :, y1:y2, x1:x2] = imgs[perm, :, y1:y2, x1:x2]
 
        # Adjust lambda to actual area
        lam_actual = 1 - (x2 - x1) * (y2 - y1) / (H * W)
 
        # Soft labels for defect head only
        # Store both original and mixed labels; loss will handle soft targets
        # We use a simple approach: randomly assign label based on lambda
        # (avoids changing loss function signature)
        # For samples where lam > 0.5 keep original label, else swap
        if lam_actual >= 0.5:
            imgs = imgs_copy
        else:
            imgs = imgs_copy
            def_l_new = def_l.clone()
            def_l_new[known] = def_l[perm]
            def_l = def_l_new
 
    return imgs, bin_l, def_l, prod_l, list(paths)
 
 
# ── Build dataloaders ──────────────────────────────────────────────────
BATCH_SIZE  = 32
NUM_WORKERS = 4
 
train_3head_loader = DataLoader(
    train_3head_ds,
    batch_size  = BATCH_SIZE,
    sampler     = sampler,
    num_workers = NUM_WORKERS,
    pin_memory  = True,
    drop_last   = True,
    collate_fn  = cutmix_collate,   # ← CutMix applied here
)
 
val_3head_loader = DataLoader(
    val_3head_ds,
    batch_size  = BATCH_SIZE,
    shuffle     = False,
    num_workers = NUM_WORKERS,
    pin_memory  = True,
    # No CutMix on val — clean evaluation
)
 
print(f"✅ CELL 78C COMPLETE — CutMix DataLoader")
print(f"   Train batches : {len(train_3head_loader)}")
print(f"   Val batches   : {len(val_3head_loader)}")
print(f"   CutMix        : ✅ applied in collate (p=0.5 per batch)")
print(f"   Weighted sampler: ✅ active")

✅ CELL 78C COMPLETE — CutMix DataLoader
   Train batches : 510
   Val batches   : 105
   CutMix        : ✅ applied in collate (p=0.5 per batch)
   Weighted sampler: ✅ active


In [22]:
# ======================================================================
# ENH_CELL_2B — DEFINE kd_loss  (run before ENH_CELL_3)
# ======================================================================
import torch.nn.functional as F
import torch.nn as nn

class KDLoss(nn.Module):
    def __init__(self, T_def=4.0, T_bin=2.0, T_prod=3.0, alpha=0.5):
        super().__init__()
        self.T_def=T_def; self.T_bin=T_bin; self.T_prod=T_prod; self.alpha=alpha

    def set_alpha(self, a):
        self.alpha = a

    def _kl(self, s, t, T):
        return F.kl_div(
            F.log_softmax(s / T, dim=1),
            F.softmax(t  / T, dim=1),
            reduction='batchmean') * (T * T)

kd_loss = KDLoss(T_def=4.0, T_bin=2.0, T_prod=3.0, alpha=0.5)

print("✅ ENH_CELL_2B COMPLETE — kd_loss defined")
print(f"   T_bin={kd_loss.T_bin}  T_def={kd_loss.T_def}  "
      f"T_prod={kd_loss.T_prod}  alpha={kd_loss.alpha}")

✅ ENH_CELL_2B COMPLETE — kd_loss defined
   T_bin=2.0  T_def=4.0  T_prod=3.0  alpha=0.5


In [23]:
# ======================================================================
# ENH_CELL_3 — ENHANCED TRAINING LOOP
# Losses: task_kd + λ_feat * feat_align + λ_con * supcon
# ======================================================================
import numpy as np
from sklearn.metrics import precision_recall_fscore_support

def evaluate_enh(model, loader):
    """Same evaluator as KD_CELL_5 — clean val/test pass."""
    model.eval()
    all_pred, all_true = [], []
    bin_c = prod_c = tot = def_c = def_tot = 0
    with torch.no_grad():
        for batch in loader:
            imgs, bin_lbl, def_lbl, prod_lbl, _ = batch
            imgs     = imgs.to(device)
            bin_lbl  = bin_lbl.to(device)
            def_lbl  = def_lbl.to(device)
            prod_lbl = prod_lbl.to(device)
            sb, sd, sp = model(imgs)
            bin_c  += ((torch.sigmoid(sb.squeeze()) > 0.5).long()
                       == bin_lbl).sum().item()
            prod_c += (sp.argmax(1) == prod_lbl).sum().item()
            tot    += imgs.size(0)
            known   = def_lbl != -1
            if known.sum() > 0:
                preds    = sd[known].argmax(1)
                def_c   += (preds == def_lbl[known]).sum().item()
                def_tot += known.sum().item()
                all_pred.extend(preds.cpu().tolist())
                all_true.extend(def_lbl[known].cpu().tolist())
    bin_acc  = 100. * bin_c  / max(tot,     1)
    def_acc  = 100. * def_c  / max(def_tot, 1)
    prod_acc = 100. * prod_c / max(tot,     1)
    if all_true:
        p, r, f, _ = precision_recall_fscore_support(
            all_true, all_pred,
            labels=list(range(NUM_DEFECT_TYPES)), zero_division=0)
        f1_macro = float(f.mean())
    else:
        p = r = f = np.zeros(NUM_DEFECT_TYPES); f1_macro = 0.0
    return bin_acc, def_acc, prod_acc, f1_macro, p, r, f


# ── Training ───────────────────────────────────────────────────────────
best_f1_enh = 0.0

for epoch in range(1, ENH_EPOCHS + 1):

    model.train()
    projector.train()
    teacher.eval()
    criterion_dict['multitask'].set_epoch(epoch)

    tr_bin_c = tr_def_c = tr_prod_c = tr_def_tot = tr_tot = 0
    tr_pred, tr_true = [], []

    for batch in train_3head_loader:
        imgs, bin_lbl, def_lbl, prod_lbl, _ = batch
        imgs     = imgs.to(device)
        bin_lbl  = bin_lbl.to(device)
        def_lbl  = def_lbl.to(device)
        prod_lbl = prod_lbl.to(device)

        optimizer_enh.zero_grad()

        with torch.cuda.amp.autocast():

            # ── Forward (hooks auto-populate feature dicts) ───────────
            s_bin, s_def, s_prod = model(imgs)    # hooks: pool + shared
            with torch.no_grad():
                t_bin, t_def, t_prod = teacher(imgs)  # hook: pool

            s_bin_sq = s_bin.squeeze()
            t_bin_sq = t_bin.squeeze()
            known    = def_lbl != -1

            # ── 1. Task + Logit-KD loss (same as KD_CELL_5) ──────────
            hard_bin  = criterion_dict['binary'](s_bin_sq, bin_lbl.float())
            hard_prod = criterion_dict['product'](s_prod, prod_lbl)

            if known.sum() > 0:
                hard_def = criterion_dict['defect'](
                    s_def[known], def_lbl[known])
                soft_def = kd_loss._kl(
                    s_def[known], t_def[known], kd_loss.T_def)
            else:
                hard_def = torch.tensor(0.0, device=device)
                soft_def = torch.tensor(0.0, device=device)

            s_b2 = torch.stack([-s_bin_sq, s_bin_sq], dim=1)
            t_b2 = torch.stack([-t_bin_sq, t_bin_sq], dim=1)
            soft_bin  = kd_loss._kl(s_b2,   t_b2,   kd_loss.T_bin)
            soft_prod = kd_loss._kl(s_prod, t_prod, kd_loss.T_prod)

            a  = ALPHA_FIXED
            lb = a * hard_bin  + (1 - a) * soft_bin
            ld = a * hard_def  + (1 - a) * soft_def
            lp = a * hard_prod + (1 - a) * soft_prod
            tw = criterion_dict['multitask'].weights()
            task_kd_loss = (float(tw[0])*lb +
                            float(tw[1])*ld +
                            float(tw[2])*lp)

            # ── 2. Feature alignment loss ─────────────────────────────
            # project student 960-dim → 1280-dim, align with teacher
            s_feat_proj = projector(_student_pool_feat['feat'])
            t_feat_stop = _teacher_pool_feat['feat'].detach()
            feat_loss   = cosine_feat_loss(s_feat_proj, t_feat_stop)

            # ── 3. Supervised Contrastive loss ────────────────────────
            # On 512-dim shared embeddings, defect-labeled only
            if known.sum() >= 4:
                s_emb_known = _student_shared_feat['feat'][known]
                lbl_known   = def_lbl[known]
                # Only proceed if ≥2 unique classes present in batch
                n_unique = lbl_known.unique().shape[0]
                if n_unique >= 2:
                    con_loss = supcon(s_emb_known, lbl_known)
                else:
                    con_loss = torch.tensor(0.0, device=device)
            else:
                con_loss = torch.tensor(0.0, device=device)

            # ── Combined loss ─────────────────────────────────────────
            total_loss = (task_kd_loss
                          + LAMBDA_FEAT * feat_loss
                          + LAMBDA_CON  * con_loss)

        scaler_enh.scale(total_loss).backward()
        scaler_enh.unscale_(optimizer_enh)
        torch.nn.utils.clip_grad_norm_(
            list(model.parameters()) + list(projector.parameters()), 1.0)
        scaler_enh.step(optimizer_enh)
        scaler_enh.update()

        # ── Accumulate train metrics ──────────────────────────────────
        tr_tot    += imgs.size(0)
        tr_bin_c  += ((torch.sigmoid(s_bin_sq.detach()) > 0.5).long()
                      == bin_lbl).sum().item()
        tr_prod_c += (s_prod.detach().argmax(1) == prod_lbl).sum().item()
        if known.sum() > 0:
            preds = s_def.detach()[known].argmax(1)
            tr_def_c   += (preds == def_lbl[known]).sum().item()
            tr_def_tot += known.sum().item()
            tr_pred.extend(preds.cpu().tolist())
            tr_true.extend(def_lbl[known].cpu().tolist())

    scheduler_enh.step()

    # ── Train metrics ─────────────────────────────────────────────────
    t_bin_acc  = 100. * tr_bin_c  / max(tr_tot,     1)
    t_def_acc  = 100. * tr_def_c  / max(tr_def_tot, 1)
    t_prod_acc = 100. * tr_prod_c / max(tr_tot,     1)
    if tr_true:
        _, _, tf, _ = precision_recall_fscore_support(
            tr_true, tr_pred,
            labels=list(range(NUM_DEFECT_TYPES)), zero_division=0)
        t_f1 = float(tf.mean())
    else:
        t_f1 = 0.0

    # ── Validation ────────────────────────────────────────────────────
    v_bin, v_def, v_prod, v_f1, vp, vr, vf = evaluate_enh(
        model, val_3head_loader)

    cur_lr = optimizer_enh.param_groups[1]['lr']
    tw     = criterion_dict['multitask'].weights()

    # ── Print ─────────────────────────────────────────────────────────
    print(f"\nEpoch [{epoch}/{ENH_EPOCHS}]  LR={cur_lr:.1e}  "
          f"TaskW=[bin={float(tw[0]):.2f} "
          f"def={float(tw[1]):.2f} "
          f"prod={float(tw[2]):.2f}]  [FINETUNE+FEAT+CON]")
    print()
    print(f"  Train → Bin={t_bin_acc:.1f}%  Def={t_def_acc:.1f}%  "
          f"F1={t_f1:.3f}  Prod={t_prod_acc:.1f}%")
    print(f"  Val   → Bin={v_bin:.1f}%  Def={v_def:.1f}%  "
          f"F1={v_f1:.3f}  Prod={v_prod:.1f}%")
    print()
    print(f"  {'Class':<22} {'P':>6}  {'R':>6}  {'F1':>6}")
    print(f"  {'-'*40}")
    for i, name in enumerate(SEMANTIC_GROUPS):
        print(f"  {name:<22} {vp[i]:>6.3f}  {vr[i]:>6.3f}  {vf[i]:>6.3f}")
    print(f"  {'-'*40}")
    print(f"  {'MACRO':<22} {float(np.mean(vp)):>6.3f}  "
          f"{float(np.mean(vr)):>6.3f}  {v_f1:>6.3f}")

    # ── Save best ──────────────────────────────────────────────────────
    if v_f1 > best_f1_enh:
        best_f1_enh = v_f1
        torch.save({
            'epoch':               epoch,
            'model_state_dict':    model.state_dict(),
            'projector_state':     projector.state_dict(),
            'multitask_state':     criterion_dict['multitask'].state_dict()
                                   if hasattr(criterion_dict['multitask'],
                                              'state_dict') else None,
            'optimizer_state':     optimizer_enh.state_dict(),
            'val_score':           v_f1,
            'val_metrics': {
                'binary_acc':      v_bin,
                'defect_acc':      v_def,
                'defect_f1_macro': v_f1,
                'product_acc':     v_prod,
            },
            'defect_type_to_idx':  DEFECT_TYPE_TO_IDX,
            'product_type_to_idx': PRODUCT_TYPE_TO_IDX,
            'idx_to_defect_type':  IDX_TO_DEFECT_TYPE,
            'idx_to_product_type': IDX_TO_PRODUCT_TYPE,
            'lambda_feat':         LAMBDA_FEAT,
            'lambda_con':          LAMBDA_CON,
        }, ENH_SAVE_PATH)
        print(f"\n  ✅ NEW BEST  F1={best_f1_enh:.3f} → saved EdgeNet_V2_ENH.pth")

# ── Cleanup hooks ──────────────────────────────────────────────────────
h1.remove(); h2.remove(); h3.remove()

print(f"\n{'='*60}")
print(f"ENH TRAINING COMPLETE")
print(f"  Best Val F1 (macro) : {best_f1_enh:.3f}")
print(f"  Saved to            : {ENH_SAVE_PATH}")
print(f"{'='*60}")
print("✅ ENH_CELL_3 DONE")


Epoch [1/30]  LR=5.0e-05  TaskW=[bin=1.00 def=1.00 prod=1.00]  [FINETUNE+FEAT+CON]

  Train → Bin=99.5%  Def=84.2%  F1=0.843  Prod=97.5%
  Val   → Bin=98.7%  Def=89.5%  F1=0.880  Prod=99.8%

  Class                       P       R      F1
  ----------------------------------------
  contamination           0.861   0.939   0.899
  cut                     0.933   0.824   0.875
  deformation             0.846   0.733   0.786
  fracture                0.889   0.930   0.909
  hole_void               0.962   0.944   0.953
  minor_defect            0.765   0.929   0.839
  scratch                 0.920   0.793   0.852
  surface_quality         0.966   0.889   0.926
  ----------------------------------------
  MACRO                   0.893   0.873   0.880

  ✅ NEW BEST  F1=0.880 → saved EdgeNet_V2_ENH.pth

Epoch [2/30]  LR=4.9e-05  TaskW=[bin=1.00 def=1.00 prod=1.00]  [FINETUNE+FEAT+CON]

  Train → Bin=99.4%  Def=84.1%  F1=0.843  Prod=97.6%
  Val   → Bin=98.8%  Def=88.2%  F1=0.860  Prod=99.9%


In [24]:
# ======================================================================
# ENH_CELL_4 — FINAL 3-WAY TEST EVALUATION
# EdgeNet-V2  vs  EdgeNet-V2+KD  vs  EdgeNet-V2+ENH  vs  Teacher
# ======================================================================
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix
import numpy as np
from torch.utils.data import DataLoader

test_3head_loader = DataLoader(
    test_3head_ds, batch_size=32, shuffle=False,
    num_workers=4, pin_memory=True)

def test_eval(model, loader, name):
    model.eval()
    all_pred, all_true = [], []
    bin_c = prod_c = tot = def_c = def_tot = 0
    with torch.no_grad():
        for batch in loader:
            imgs, bin_lbl, def_lbl, prod_lbl, _ = batch
            imgs     = imgs.to(device)
            bin_lbl  = bin_lbl.to(device)
            def_lbl  = def_lbl.to(device)
            prod_lbl = prod_lbl.to(device)
            sb, sd, sp = model(imgs)
            bin_c  += ((torch.sigmoid(sb.squeeze()) > 0.5).long()
                       == bin_lbl).sum().item()
            prod_c += (sp.argmax(1) == prod_lbl).sum().item()
            tot    += imgs.size(0)
            known   = def_lbl != -1
            if known.sum() > 0:
                preds    = sd[known].argmax(1)
                def_c   += (preds == def_lbl[known]).sum().item()
                def_tot += known.sum().item()
                all_pred.extend(preds.cpu().tolist())
                all_true.extend(def_lbl[known].cpu().tolist())
    bin_acc  = 100. * bin_c  / max(tot,     1)
    def_acc  = 100. * def_c  / max(def_tot, 1)
    prod_acc = 100. * prod_c / max(tot,     1)
    p, r, f, s = precision_recall_fscore_support(
        all_true, all_pred,
        labels=list(range(NUM_DEFECT_TYPES)), zero_division=0)
    return dict(name=name, bin=bin_acc, defect=def_acc,
                prod=prod_acc, f1=float(f.mean()),
                p=p, r=r, f=f, s=s)


# ── Load and evaluate all models ──────────────────────────────────────
print("Evaluating all models on test set...\n")

# 1. Original EdgeNet-V2
ck_orig = torch.load('/home/sufi/training_results/models/EdgeNet_V2.pth',
                     map_location=device, weights_only=False)
model.load_state_dict(ck_orig['model_state_dict'])
r_orig = test_eval(model, test_3head_loader, "EdgeNet-V2")
print(f"  ✅ EdgeNet-V2         "
      f"Def={r_orig['defect']:.2f}%  F1={r_orig['f1']:.3f}")

# 2. EdgeNet-V2+KD
ck_kd = torch.load('/home/sufi/training_results/models/EdgeNet_V2_KD.pth',
                   map_location=device, weights_only=False)
model.load_state_dict(ck_kd['model_state_dict'])
r_kd = test_eval(model, test_3head_loader, "EdgeNet-V2+KD")
print(f"  ✅ EdgeNet-V2+KD      "
      f"Def={r_kd['defect']:.2f}%  F1={r_kd['f1']:.3f}")

# 3. EdgeNet-V2+ENH
ck_enh = torch.load('/home/sufi/training_results/models/EdgeNet_V2_ENH.pth',
                    map_location=device, weights_only=False)
model.load_state_dict(ck_enh['model_state_dict'])
r_enh = test_eval(model, test_3head_loader, "EdgeNet-V2+ENH")
print(f"  ✅ EdgeNet-V2+ENH     "
      f"Def={r_enh['defect']:.2f}%  F1={r_enh['f1']:.3f}")

# 4. Teacher
r_tea = test_eval(teacher, test_3head_loader, "EfficientNet-B0")
print(f"  ✅ EfficientNet-B0    "
      f"Def={r_tea['defect']:.2f}%  F1={r_tea['f1']:.3f}")


# ══════════════════════════════════════════════════════════════════════
# THESIS COMPARISON TABLE
# ══════════════════════════════════════════════════════════════════════
print()
print("=" * 72)
print("FINAL TEST SET — THESIS COMPARISON TABLE")
print("=" * 72)
header = (f"  {'Model':<28} {'Params':>7}  "
          f"{'Binary':>8}  {'Defect':>8}  {'F1':>7}  {'Product':>9}")
div    = "  " + "-" * 68
print(header); print(div)

BASELINES = [
    ("MobileNetV3-Small",  "—",      "—",     "79.05%", "—",     "—"),
    ("ResNet-18",          "—",      "—",     "84.46%", "—",     "—"),
    ("ResNet-50",          "—",      "—",     "90.88%", "—",     "—"),
    ("MobileNetV3-Large",  "—",      "—",     "91.22%", "—",     "—"),
]
for name, _, _, defect, _, _ in BASELINES:
    print(f"  {name:<28} {'—':>7}  {'—':>8}  {defect:>8}  {'—':>7}  {'—':>9}")
print(div)

def row(r, params, tag=""):
    print(f"  {r['name']+tag:<28} {params:>7}  "
          f"{r['bin']:>7.2f}%  {r['defect']:>7.2f}%  "
          f"{r['f1']:>7.3f}  {r['prod']:>8.2f}%")

row(r_tea,  "5.00M", "  ← Teacher"); print(div)
row(r_orig, "3.89M")
row(r_kd,   "3.89M", "  ← +KD")
row(r_enh,  "3.89M", "  ← +KD+Feat+SupCon  ★ Best")
print("=" * 72)


# ══════════════════════════════════════════════════════════════════════
# PER-CLASS F1 PROGRESSION
# ══════════════════════════════════════════════════════════════════════
print()
print("=" * 80)
print("PER-CLASS F1 PROGRESSION (Test Set)")
print("=" * 80)
print(f"  {'Class':<22}  {'Baseline':>10}  "
      f"{'KD':>10}  {'ENH':>10}  {'Gain vs KD':>12}  {'Support':>8}")
print("  " + "-" * 72)
for i, name in enumerate(SEMANTIC_GROUPS):
    f0 = r_orig['f'][i]
    fk = r_kd['f'][i]
    fe = r_enh['f'][i]
    dg = fe - fk
    sup = int(r_enh['s'][i])
    flag = "  ▲" if dg > 0.02 else ("  ▼" if dg < -0.02 else "")
    print(f"  {name:<22}  {f0:>10.3f}  {fk:>10.3f}  "
          f"{fe:>10.3f}  {dg:>+12.3f}  {sup:>8}{flag}")
print("  " + "-" * 72)
print(f"  {'MACRO':<22}  {r_orig['f1']:>10.3f}  "
      f"{r_kd['f1']:>10.3f}  {r_enh['f1']:>10.3f}  "
      f"{r_enh['f1']-r_kd['f1']:>+12.3f}")

print("=" * 80)


# ══════════════════════════════════════════════════════════════════════
# FINAL SUMMARY
# ══════════════════════════════════════════════════════════════════════
print()
print("=" * 60)
print("KNOWLEDGE TRANSFER SUMMARY")
print("=" * 60)
gap_total  = r_tea['defect'] - r_orig['defect']
gap_kd     = r_kd['defect']  - r_orig['defect']
gap_enh    = r_enh['defect'] - r_orig['defect']
print(f"  Teacher                : {r_tea['defect']:.2f}%")
print(f"  Baseline (no KD)       : {r_orig['defect']:.2f}%  (gap to teacher: "
      f"{gap_total:.2f}%)")
print(f"  + Logit KD             : {r_kd['defect']:.2f}%  "
      f"(+{gap_kd:.2f}%, closed {100*gap_kd/max(gap_total,1e-6):.1f}%)")
print(f"  + Feat align + SupCon  : {r_enh['defect']:.2f}%  "
      f"(+{gap_enh:.2f}%, closed {100*gap_enh/max(gap_total,1e-6):.1f}%)")
print(f"  Param savings vs teacher: "
      f"{(1-3.89/5.00)*100:.1f}%  (3.89M vs 5.00M)")
print(f"  ENH best epoch         : {ck_enh['epoch']}")
print("=" * 60)
print("\n✅ ENH_CELL_4 DONE — Full thesis evaluation complete")

Evaluating all models on test set...

  ✅ EdgeNet-V2         Def=85.81%  F1=0.824
  ✅ EdgeNet-V2+KD      Def=89.19%  F1=0.865
  ✅ EdgeNet-V2+ENH     Def=89.86%  F1=0.864
  ✅ EfficientNet-B0    Def=91.22%  F1=0.897

FINAL TEST SET — THESIS COMPARISON TABLE
  Model                         Params    Binary    Defect       F1    Product
  --------------------------------------------------------------------
  MobileNetV3-Small                  —         —    79.05%        —          —
  ResNet-18                          —         —    84.46%        —          —
  ResNet-50                          —         —    90.88%        —          —
  MobileNetV3-Large                  —         —    91.22%        —          —
  --------------------------------------------------------------------
  EfficientNet-B0  ← Teacher     5.00M    98.68%    91.22%    0.897     99.76%
  --------------------------------------------------------------------
  EdgeNet-V2                     3.89M    97.54%    85.81

In [25]:
# ======================================================================
# BOOST_CELL_1 — TTA + ENSEMBLE  (zero training, instant gains)
# Tests 4 variants: ENH | ENH+TTA | ENH+KD_Ensemble | ENH+TTA+Ensemble
# ======================================================================
import torch, numpy as np
from sklearn.metrics import precision_recall_fscore_support
from torch.utils.data import DataLoader

# ── Reload test loader (224×224) ──────────────────────────────────────
test_loader_224 = DataLoader(
    test_3head_ds, batch_size=32, shuffle=False,
    num_workers=4, pin_memory=True)

# ── Load ENH model ────────────────────────────────────────────────────
ck_enh = torch.load(
    '/home/sufi/training_results/models/EdgeNet_V2_ENH.pth',
    map_location=device, weights_only=False)
model.load_state_dict(ck_enh['model_state_dict'])
model.eval()
print(f"ENH loaded  : epoch {ck_enh['epoch']}  "
      f"val F1={ck_enh['val_metrics']['defect_f1_macro']:.3f}")

# ── Load KD model into a second instance ──────────────────────────────
model_kd2 = EdgeNetV2(
    num_defect  = NUM_DEFECT_TYPES,
    num_product = NUM_PRODUCT_TYPES).to(device)
ck_kd2 = torch.load(
    '/home/sufi/training_results/models/EdgeNet_V2_KD.pth',
    map_location=device, weights_only=False)
model_kd2.load_state_dict(ck_kd2['model_state_dict'])
model_kd2.eval()
print(f"KD  loaded  : epoch {ck_kd2['epoch']}  "
      f"val F1={ck_kd2['val_metrics']['defect_f1_macro']:.3f}")

# ── TTA augmentations (all tensor-level, no PIL needed) ───────────────
TTA_AUGS = [
    lambda x: x,                              # original
    lambda x: torch.flip(x, [3]),             # horizontal flip
    lambda x: torch.flip(x, [2]),             # vertical flip
    lambda x: torch.rot90(x, 1, [2, 3]),      # rotate 90°
    lambda x: torch.rot90(x, 3, [2, 3]),      # rotate 270°
]

def tta_pred_single(mdl, imgs):
    """Run TTA on one model, return averaged probabilities."""
    bin_p, def_p, prod_p = [], [], []
    for aug in TTA_AUGS:
        sb, sd, sp = mdl(aug(imgs))
        bin_p.append(torch.sigmoid(sb.squeeze(1)))
        def_p.append(torch.softmax(sd, dim=1))
        prod_p.append(torch.softmax(sp, dim=1))
    return (torch.stack(bin_p).mean(0),
            torch.stack(def_p).mean(0),
            torch.stack(prod_p).mean(0))

def evaluate_boosted(model_list, loader, use_tta=False, label=""):
    """
    model_list : list of (model, weight) — weights must sum to 1.0
    use_tta    : apply 5-aug TTA per model
    """
    for m, _ in model_list:
        m.eval()

    all_pred, all_true = [], []
    bin_c = prod_c = tot = def_c = def_tot = 0

    with torch.no_grad():
        for batch in loader:
            imgs, bin_lbl, def_lbl, prod_lbl, _ = batch
            imgs     = imgs.to(device)
            bin_lbl  = bin_lbl.to(device)
            def_lbl  = def_lbl.to(device)
            prod_lbl = prod_lbl.to(device)

            B = imgs.size(0)
            avg_bin  = torch.zeros(B,                  device=device)
            avg_def  = torch.zeros(B, NUM_DEFECT_TYPES, device=device)
            avg_prod = torch.zeros(B, NUM_PRODUCT_TYPES,device=device)

            for mdl, w in model_list:
                if use_tta:
                    bp, dp, pp = tta_pred_single(mdl, imgs)
                else:
                    sb, sd, sp = mdl(imgs)
                    bp = torch.sigmoid(sb.squeeze(1))
                    dp = torch.softmax(sd, dim=1)
                    pp = torch.softmax(sp, dim=1)
                avg_bin  += w * bp
                avg_def  += w * dp
                avg_prod += w * pp

            bin_c  += ((avg_bin > 0.5).long() == bin_lbl).sum().item()
            prod_c += (avg_prod.argmax(1) == prod_lbl).sum().item()
            tot    += B
            known   = def_lbl != -1
            if known.sum() > 0:
                preds    = avg_def[known].argmax(1)
                def_c   += (preds == def_lbl[known]).sum().item()
                def_tot += known.sum().item()
                all_pred.extend(preds.cpu().tolist())
                all_true.extend(def_lbl[known].cpu().tolist())

    bin_acc  = 100. * bin_c  / max(tot,     1)
    def_acc  = 100. * def_c  / max(def_tot, 1)
    prod_acc = 100. * prod_c / max(tot,     1)
    p, r, f, s = precision_recall_fscore_support(
        all_true, all_pred,
        labels=list(range(NUM_DEFECT_TYPES)), zero_division=0)
    f1 = float(f.mean())
    print(f"  {label:<40} Def={def_acc:.2f}%  F1={f1:.3f}  "
          f"Bin={bin_acc:.2f}%  Prod={prod_acc:.2f}%")
    return dict(label=label, bin=bin_acc, defect=def_acc,
                prod=prod_acc, f1=f1, p=p, r=r, f=f, s=s)


# ── Run all 4 zero-cost variants ──────────────────────────────────────
print("\n" + "="*70)
print("ZERO-COST BOOST RESULTS  (Test Set, 224×224)")
print("="*70)

r_enh_plain = evaluate_boosted(
    [(model, 1.0)], test_loader_224,
    use_tta=False, label="ENH  (no boost)")

r_enh_tta = evaluate_boosted(
    [(model, 1.0)], test_loader_224,
    use_tta=True,  label="ENH + TTA (5 augs)")

r_ensemble = evaluate_boosted(
    [(model, 0.6), (model_kd2, 0.4)], test_loader_224,
    use_tta=False, label="ENH×0.6 + KD×0.4 Ensemble")

r_ens_tta = evaluate_boosted(
    [(model, 0.6), (model_kd2, 0.4)], test_loader_224,
    use_tta=True,  label="ENH×0.6 + KD×0.4 + TTA ★")

print("="*70)
best_zero = max([r_enh_plain, r_enh_tta, r_ensemble, r_ens_tta],
                key=lambda x: x['f1'])
print(f"\n  Best zero-cost F1 : {best_zero['f1']:.3f}  "
      f"({best_zero['label']})")
print("\n✅ BOOST_CELL_1 DONE")

ENH loaded  : epoch 23  val F1=0.896
KD  loaded  : epoch 88  val F1=0.882

ZERO-COST BOOST RESULTS  (Test Set, 224×224)
  ENH  (no boost)                          Def=89.86%  F1=0.864  Bin=98.32%  Prod=99.76%
  ENH + TTA (5 augs)                       Def=89.86%  F1=0.870  Bin=98.62%  Prod=99.82%
  ENH×0.6 + KD×0.4 Ensemble                Def=89.86%  F1=0.862  Bin=98.38%  Prod=99.76%
  ENH×0.6 + KD×0.4 + TTA ★                 Def=89.86%  F1=0.870  Bin=98.62%  Prod=99.82%

  Best zero-cost F1 : 0.870  (ENH + TTA (5 augs))

✅ BOOST_CELL_1 DONE


In [26]:
# ======================================================================
# HR256_CELL_1 — 256×256 DATALOADERS
# ======================================================================
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

hr_train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(0.3, 0.3, 0.2, 0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

hr_val_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

# ── Dataset instances ─────────────────────────────────────────────────
hr_train_ds = ThreeHeadDataset(train_df, transform=hr_train_transform)
hr_val_ds   = ThreeHeadDataset(val_df,   transform=hr_val_transform)
hr_test_ds  = ThreeHeadDataset(test_df,  transform=hr_val_transform)

# ── Sampler (reuse existing function from Cell 78C) ───────────────────
hr_sampler = make_weighted_sampler(train_df)

# ── Dataloaders (batch_size=24 — larger images need less per batch) ───
HR_BATCH = 24

hr_train_loader = DataLoader(
    hr_train_ds,
    batch_size  = HR_BATCH,
    sampler     = hr_sampler,
    num_workers = 4,
    pin_memory  = True,
    drop_last   = True,
    collate_fn  = cutmix_collate,
)
hr_val_loader = DataLoader(
    hr_val_ds,
    batch_size  = HR_BATCH,
    shuffle     = False,
    num_workers = 4,
    pin_memory  = True,
)
hr_test_loader = DataLoader(
    hr_test_ds,
    batch_size  = HR_BATCH,
    shuffle     = False,
    num_workers = 4,
    pin_memory  = True,
)

print("✅ HR256_CELL_1 COMPLETE — 256×256 dataloaders ready")
print(f"   Train batches : {len(hr_train_loader)}")
print(f"   Val batches   : {len(hr_val_loader)}")
print(f"   Test batches  : {len(hr_test_loader)}")
print(f"   Batch size    : {HR_BATCH}")

✅ HR256_CELL_1 COMPLETE — 256×256 dataloaders ready
   Train batches : 680
   Val batches   : 140
   Test batches  : 70
   Batch size    : 24


In [27]:
# ======================================================================
# HR256_CELL_2 — HR256 FINE-TUNE  (15 epochs from ENH checkpoint)
# Starts from ENH (F1=0.896 val), adapts to 256×256 resolution
# Saves to: /models/HR256/EdgeNet_V2_ENH_HR256.pth
# ======================================================================
from pathlib import Path
from sklearn.metrics import precision_recall_fscore_support
from torch.cuda.amp import GradScaler
import numpy as np

# ── Save path (separate folder, never overwrites ENH) ─────────────────
HR256_DIR  = Path('/home/sufi/training_results/models/HR256')
HR256_DIR.mkdir(parents=True, exist_ok=True)
HR256_SAVE = HR256_DIR / 'EdgeNet_V2_ENH_HR256.pth'

HR256_EPOCHS = 15

# ── Load from ENH best checkpoint ─────────────────────────────────────
ck_enh = torch.load(
    '/home/sufi/training_results/models/EdgeNet_V2_ENH.pth',
    map_location=device, weights_only=False)
model.load_state_dict(ck_enh['model_state_dict'])
model.freeze_backbone(False)
print(f"Loaded ENH epoch {ck_enh['epoch']}  "
      f"val F1={ck_enh['val_metrics']['defect_f1_macro']:.3f}")

# ── Optimizer — very low LR, resolution adaptation only ───────────────
bb_params    = list(model.early_features.parameters()) + \
               list(model.late_features.parameters())
coord_params = list(model.coord_att1.parameters()) + \
               list(model.coord_att2.parameters())
head_params  = list(model.shared.parameters())      + \
               list(model.binary_head.parameters()) + \
               list(model.defect_head.parameters()) + \
               list(model.product_head.parameters())

optimizer_hr = torch.optim.AdamW([
    {'params': bb_params,    'lr': 5e-6},
    {'params': coord_params, 'lr': 2e-5},
    {'params': head_params,  'lr': 2e-5},
], weight_decay=1e-4)

scheduler_hr = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_hr, T_max=HR256_EPOCHS, eta_min=1e-7)
scaler_hr    = GradScaler()

# ── Evaluation function ────────────────────────────────────────────────
def evaluate_hr(model, loader):
    model.eval()
    all_pred, all_true = [], []
    bin_c = prod_c = tot = def_c = def_tot = 0
    with torch.no_grad():
        for batch in loader:
            imgs, bin_lbl, def_lbl, prod_lbl, _ = batch
            imgs     = imgs.to(device)
            bin_lbl  = bin_lbl.to(device)
            def_lbl  = def_lbl.to(device)
            prod_lbl = prod_lbl.to(device)
            sb, sd, sp = model(imgs)
            bin_c  += ((torch.sigmoid(sb.squeeze()) > 0.5).long()
                       == bin_lbl).sum().item()
            prod_c += (sp.argmax(1) == prod_lbl).sum().item()
            tot    += imgs.size(0)
            known   = def_lbl != -1
            if known.sum() > 0:
                preds    = sd[known].argmax(1)
                def_c   += (preds == def_lbl[known]).sum().item()
                def_tot += known.sum().item()
                all_pred.extend(preds.cpu().tolist())
                all_true.extend(def_lbl[known].cpu().tolist())
    bin_acc  = 100. * bin_c  / max(tot,     1)
    def_acc  = 100. * def_c  / max(def_tot, 1)
    prod_acc = 100. * prod_c / max(tot,     1)
    p, r, f, _ = precision_recall_fscore_support(
        all_true, all_pred,
        labels=list(range(NUM_DEFECT_TYPES)), zero_division=0)
    return bin_acc, def_acc, prod_acc, float(f.mean()), p, r, f

# ── Training loop ──────────────────────────────────────────────────────
best_f1_hr = 0.0

for epoch in range(1, HR256_EPOCHS + 1):

    model.train()
    criterion_dict['multitask'].set_epoch(epoch)

    tr_bin_c = tr_def_c = tr_prod_c = tr_def_tot = tr_tot = 0
    tr_pred, tr_true = [], []

    for batch in hr_train_loader:
        imgs, bin_lbl, def_lbl, prod_lbl, _ = batch
        imgs     = imgs.to(device)
        bin_lbl  = bin_lbl.to(device)
        def_lbl  = def_lbl.to(device)
        prod_lbl = prod_lbl.to(device)

        optimizer_hr.zero_grad()

        with torch.cuda.amp.autocast():
            s_bin, s_def, s_prod = model(imgs)
            s_bin_sq = s_bin.squeeze()
            known    = def_lbl != -1

            # Hard task losses only (resolution adaptation)
            hard_bin  = criterion_dict['binary'](s_bin_sq, bin_lbl.float())
            hard_prod = criterion_dict['product'](s_prod, prod_lbl)
            hard_def  = criterion_dict['defect'](
                s_def[known], def_lbl[known]) if known.sum() > 0 \
                else torch.tensor(0.0, device=device)

            tw         = criterion_dict['multitask'].weights()
            total_loss = (float(tw[0]) * hard_bin  +
                          float(tw[1]) * hard_def  +
                          float(tw[2]) * hard_prod)

        scaler_hr.scale(total_loss).backward()
        scaler_hr.unscale_(optimizer_hr)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler_hr.step(optimizer_hr)
        scaler_hr.update()

        tr_tot    += imgs.size(0)
        tr_bin_c  += ((torch.sigmoid(s_bin_sq.detach()) > 0.5).long()
                      == bin_lbl).sum().item()
        tr_prod_c += (s_prod.detach().argmax(1) == prod_lbl).sum().item()
        if known.sum() > 0:
            preds = s_def.detach()[known].argmax(1)
            tr_def_c   += (preds == def_lbl[known]).sum().item()
            tr_def_tot += known.sum().item()
            tr_pred.extend(preds.cpu().tolist())
            tr_true.extend(def_lbl[known].cpu().tolist())

    scheduler_hr.step()

    t_bin_acc  = 100. * tr_bin_c  / max(tr_tot,     1)
    t_def_acc  = 100. * tr_def_c  / max(tr_def_tot, 1)
    t_prod_acc = 100. * tr_prod_c / max(tr_tot,     1)
    if tr_true:
        _, _, tf, _ = precision_recall_fscore_support(
            tr_true, tr_pred,
            labels=list(range(NUM_DEFECT_TYPES)), zero_division=0)
        t_f1 = float(tf.mean())
    else:
        t_f1 = 0.0

    v_bin, v_def, v_prod, v_f1, vp, vr, vf = evaluate_hr(
        model, hr_val_loader)
    cur_lr = optimizer_hr.param_groups[1]['lr']
    tw     = criterion_dict['multitask'].weights()

    print(f"\nEpoch [{epoch}/{HR256_EPOCHS}]  LR={cur_lr:.1e}  "
          f"TaskW=[bin={float(tw[0]):.2f} def={float(tw[1]):.2f} "
          f"prod={float(tw[2]):.2f}]  [HR256]")
    print()
    print(f"  Train → Bin={t_bin_acc:.1f}%  Def={t_def_acc:.1f}%  "
          f"F1={t_f1:.3f}  Prod={t_prod_acc:.1f}%")
    print(f"  Val   → Bin={v_bin:.1f}%  Def={v_def:.1f}%  "
          f"F1={v_f1:.3f}  Prod={v_prod:.1f}%")
    print()
    print(f"  {'Class':<22} {'P':>6}  {'R':>6}  {'F1':>6}")
    print(f"  {'-'*40}")
    for i, name in enumerate(SEMANTIC_GROUPS):
        print(f"  {name:<22} {vp[i]:>6.3f}  {vr[i]:>6.3f}  {vf[i]:>6.3f}")
    print(f"  {'-'*40}")
    print(f"  {'MACRO':<22} {float(np.mean(vp)):>6.3f}  "
          f"{float(np.mean(vr)):>6.3f}  {v_f1:>6.3f}")

    if v_f1 > best_f1_hr:
        best_f1_hr = v_f1
        torch.save({
            'epoch':               epoch,
            'model_state_dict':    model.state_dict(),
            'multitask_state':     criterion_dict['multitask'].state_dict()
                                   if hasattr(criterion_dict['multitask'],
                                              'state_dict') else None,
            'optimizer_state':     optimizer_hr.state_dict(),
            'val_score':           v_f1,
            'val_metrics': {
                'binary_acc':      v_bin,
                'defect_acc':      v_def,
                'defect_f1_macro': v_f1,
                'product_acc':     v_prod,
            },
            'input_size':          256,
            'parent_checkpoint':   'EdgeNet_V2_ENH.pth',
            'defect_type_to_idx':  DEFECT_TYPE_TO_IDX,
            'product_type_to_idx': PRODUCT_TYPE_TO_IDX,
            'idx_to_defect_type':  IDX_TO_DEFECT_TYPE,
            'idx_to_product_type': IDX_TO_PRODUCT_TYPE,
        }, HR256_SAVE)
        print(f"\n  ✅ NEW BEST  F1={best_f1_hr:.3f} "
              f"→ saved HR256/EdgeNet_V2_ENH_HR256.pth")

print(f"\n{'='*60}")
print(f"HR256 TRAINING COMPLETE")
print(f"  Best Val F1  : {best_f1_hr:.3f}")
print(f"  Saved to     : {HR256_SAVE}")
print(f"{'='*60}")
print("✅ HR256_CELL_2 DONE")

Loaded ENH epoch 23  val F1=0.896

Epoch [1/15]  LR=2.0e-05  TaskW=[bin=1.00 def=1.00 prod=1.00]  [HR256]

  Train → Bin=99.0%  Def=84.2%  F1=0.841  Prod=96.9%
  Val   → Bin=98.4%  Def=88.2%  F1=0.854  Prod=99.9%

  Class                       P       R      F1
  ----------------------------------------
  contamination           0.938   0.909   0.923
  cut                     0.824   0.824   0.824
  deformation             0.818   0.600   0.692
  fracture                0.886   0.907   0.897
  hole_void               1.000   0.944   0.971
  minor_defect            0.727   0.952   0.825
  scratch                 0.815   0.759   0.786
  surface_quality         0.949   0.889   0.918
  ----------------------------------------
  MACRO                   0.870   0.848   0.854

  ✅ NEW BEST  F1=0.854 → saved HR256/EdgeNet_V2_ENH_HR256.pth

Epoch [2/15]  LR=1.9e-05  TaskW=[bin=1.00 def=1.00 prod=1.00]  [HR256]

  Train → Bin=99.0%  Def=85.2%  F1=0.851  Prod=97.0%
  Val   → Bin=98.5%  Def=89.5% 

In [2]:
# ======================================================================
# SETUP_CELL — COMPLETE ENVIRONMENT  (run this FIRST in new notebook)
# ======================================================================
import os, gc, torch, numpy as np, pandas as pd, cv2
from pathlib import Path
from PIL import Image
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import precision_recall_fscore_support

# ── Device ────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# ── Globals ───────────────────────────────────────────────────────────
SEMANTIC_GROUPS = [
    'contamination','cut','deformation','fracture',
    'hole_void','minor_defect','scratch','surface_quality'
]
NUM_DEFECT_TYPES  = 8
NUM_PRODUCT_TYPES = 17
DEFECT_TYPE_TO_IDX = {n:i for i,n in enumerate(SEMANTIC_GROUPS)}
IDX_TO_DEFECT_TYPE = {i:n for i,n in enumerate(SEMANTIC_GROUPS)}

# ── CoordAttention ────────────────────────────────────────────────────
class CoordAttention(nn.Module):
    def __init__(self, channels, reduction=32):
        super().__init__()
        mid = max(8, channels // reduction)
        self.conv1  = nn.Conv2d(channels, mid, 1, bias=False)
        self.bn1    = nn.BatchNorm2d(mid)
        self.act    = nn.ReLU()
        self.conv_h = nn.Conv2d(mid, channels, 1, bias=False)
        self.conv_w = nn.Conv2d(mid, channels, 1, bias=False)

    def forward(self, x):
        B, C, H, W = x.shape
        x_h = x.mean(dim=3, keepdim=True)
        x_w = x.mean(dim=2, keepdim=True).permute(0,1,3,2)
        y   = torch.cat([x_h, x_w], dim=2)
        y   = self.act(self.bn1(self.conv1(y)))
        x_h, x_w = torch.split(y, [H, W], dim=2)
        x_w = x_w.permute(0,1,3,2)
        return x * torch.sigmoid(self.conv_h(x_h)) \
                 * torch.sigmoid(self.conv_w(x_w))

# ── EdgeNetV2 ─────────────────────────────────────────────────────────
class EdgeNetV2(nn.Module):
    def __init__(self, num_defect=8, num_product=17):
        super().__init__()
        base = models.mobilenet_v3_large(
            weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V2)
        self.early_features = base.features[:7]
        self.coord_att1     = CoordAttention(40)
        self.late_features  = base.features[7:]
        self.coord_att2     = CoordAttention(960)
        self.pool           = nn.AdaptiveAvgPool2d(1)
        self.shared = nn.Sequential(
            nn.Linear(960, 512), nn.Hardswish(), nn.Dropout(0.2))
        self.binary_head = nn.Sequential(
            nn.Linear(512,128), nn.Hardswish(),
            nn.Dropout(0.2), nn.Linear(128,1))
        self.defect_head = nn.Sequential(
            nn.Linear(512,256), nn.Hardswish(),
            nn.Dropout(0.3), nn.Linear(256,num_defect))
        self.product_head = nn.Sequential(
            nn.Linear(512,256), nn.Hardswish(),
            nn.Dropout(0.2), nn.Linear(256,num_product))

    def freeze_backbone(self, freeze=True):
        for p in self.early_features.parameters():
            p.requires_grad = not freeze
        for p in self.late_features.parameters():
            p.requires_grad = not freeze

    def count_params(self, trainable_only=False):
        if trainable_only:
            return sum(p.numel() for p in self.parameters()
                       if p.requires_grad)
        return sum(p.numel() for p in self.parameters())

    def forward(self, x):
        x = self.early_features(x)
        x = self.coord_att1(x)
        x = self.late_features(x)
        x = self.coord_att2(x)
        x = self.pool(x).flatten(1)
        x = self.shared(x)
        return self.binary_head(x), self.defect_head(x), \
               self.product_head(x)

# ── EfficientNetTeacher ───────────────────────────────────────────────
class EfficientNetTeacher(nn.Module):
    def __init__(self, num_defect=8, num_product=17):
        super().__init__()
        base          = models.efficientnet_b0(
            weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        self.features = base.features
        self.pool     = nn.AdaptiveAvgPool2d(1)
        self.shared = nn.Sequential(
            nn.Linear(1280,512), nn.SiLU(), nn.Dropout(0.3))
        self.binary_head = nn.Sequential(
            nn.Linear(512,128), nn.ReLU(), nn.Linear(128,1))
        self.defect_head = nn.Sequential(
            nn.Linear(512,256), nn.SiLU(),
            nn.Dropout(0.2), nn.Linear(256,num_defect))
        self.product_head = nn.Sequential(
            nn.Linear(512,256), nn.SiLU(),
            nn.Linear(256,num_product))

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x).flatten(1)
        s = self.shared(x)
        return self.binary_head(s), self.defect_head(s), \
               self.product_head(s)

# ── KDLoss ────────────────────────────────────────────────────────────
class KDLoss(nn.Module):
    def __init__(self, T_def=4.0, T_bin=2.0, T_prod=3.0, alpha=0.5):
        super().__init__()
        self.T_def=T_def; self.T_bin=T_bin
        self.T_prod=T_prod; self.alpha=alpha

    def set_alpha(self, a): self.alpha = a

    def _kl(self, s, t, T):
        return F.kl_div(
            F.log_softmax(s/T, dim=1),
            F.softmax(t/T,  dim=1),
            reduction='batchmean') * (T*T)

kd_loss = KDLoss(T_def=4.0, T_bin=2.0, T_prod=3.0, alpha=0.5)

# ── ThreeHeadDataset ──────────────────────────────────────────────────
class ThreeHeadDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df        = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = cv2.imread(row['image_path'])
        if img is None:
            img = np.zeros((224,224,3), dtype=np.uint8)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = Image.fromarray(img)
        if self.transform:
            img = self.transform(img)
        return (img,
                int(row['binary_label']),
                int(row['defect_type_label']),
                int(row['product_type_label']),
                row['image_path'])

# ── Load CSV + build splits ───────────────────────────────────────────
DEFECT_GROUP_MAP = {
    'crack':'fracture','fracture':'fracture','faulty_imprint':'fracture',
    'hole':'hole_void','void':'hole_void','pit':'hole_void',
    'blowhole':'hole_void','scratch':'scratch','score':'scratch',
    'stain':'surface_quality','color':'surface_quality',
    'rough':'surface_quality','uneven':'surface_quality',
    'inclusion':'surface_quality','discolor':'surface_quality',
    'pilling':'surface_quality','bent':'deformation',
    'bent_lead':'deformation','squeeze':'deformation',
    'deformation':'deformation','contamination':'contamination',
    'glue':'contamination','oil':'contamination',
    'glue_strip':'contamination','liquid':'contamination',
    'metal_contamination':'contamination','missing':'minor_defect',
    'misplaced':'minor_defect','flip':'minor_defect',
    'missing_hole':'minor_defect','thread':'minor_defect',
    'cable_swap':'minor_defect','combined':'minor_defect',
    'cut':'cut','hole_void':'hole_void',
    'surface_quality':'surface_quality','minor_defect':'minor_defect',
}

df_3head = pd.read_csv(
    '/home/sufi/merged_dataset_metadata_augmented.csv')
if 'path'     in df_3head.columns:
    df_3head = df_3head.rename(columns={'path':'image_path'})
if 'category' in df_3head.columns:
    df_3head = df_3head.rename(columns={'category':'product_type'})

def compute_binary(v):
    if pd.isna(v): return 0
    return 0 if str(v).strip().lower() in \
        ('0','good','normal','ok','casting_ok') else 1

def remap_defect(raw):
    if pd.isna(raw): return -1
    s = str(raw).strip().lower()
    if s in ('good','normal','casting_ok',''): return -1
    sem = DEFECT_GROUP_MAP.get(s)
    if sem is None and s in SEMANTIC_GROUPS: sem = s
    if sem is None:
        for k,v in DEFECT_GROUP_MAP.items():
            if k in s: sem=v; break
    return -1 if sem is None else DEFECT_TYPE_TO_IDX.get(sem,-1)

df_3head['binary_label']     = df_3head['label'].apply(compute_binary)
df_3head['defect_type_label']= df_3head['defect_type'].apply(remap_defect)
df_3head.loc[df_3head['binary_label']==0,'defect_type_label'] = -1

all_products        = sorted(df_3head['product_type'].dropna()
                              .unique().tolist())
PRODUCT_TYPE_TO_IDX = {p:i for i,p in enumerate(all_products)}
IDX_TO_PRODUCT_TYPE = {i:p for i,p in enumerate(all_products)}
NUM_PRODUCT_TYPES   = len(all_products)

prod_labels = [PRODUCT_TYPE_TO_IDX.get(str(v),0)
               if not pd.isna(v) else 0
               for v in df_3head['product_type']]
df_3head['product_type_label'] = prod_labels

train_df = df_3head[df_3head['split']=='train'].reset_index(drop=True)
val_df   = df_3head[df_3head['split']=='val'].reset_index(drop=True)
test_df  = df_3head[df_3head['split']=='test'].reset_index(drop=True)

# ── Transforms ────────────────────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(0.3,0.3,0.2,0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
val_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

train_3head_ds = ThreeHeadDataset(train_df, transform=train_transform)
val_3head_ds   = ThreeHeadDataset(val_df,   transform=val_transform)
test_3head_ds  = ThreeHeadDataset(test_df,  transform=val_transform)

# ── Weighted sampler ──────────────────────────────────────────────────
def make_weighted_sampler(df):
    class_counts = np.ones(NUM_DEFECT_TYPES)
    for i in range(NUM_DEFECT_TYPES):
        class_counts[i] = max(1,(df['defect_type_label']==i).sum())
    cw = 1.0/class_counts; cw = cw/cw.sum()
    weights = []
    for _,row in df.iterrows():
        lbl = int(row['defect_type_label'])
        weights.append(0.3 if lbl==-1 else float(cw[lbl]))
    return WeightedRandomSampler(
        torch.tensor(weights,dtype=torch.float32),
        len(weights), replacement=True)

# ── CutMix collate ────────────────────────────────────────────────────
def cutmix_collate(batch):
    imgs,bin_l,def_l,prod_l,paths = zip(*batch)
    imgs  = torch.stack(imgs)
    bin_l = torch.tensor(bin_l,  dtype=torch.long)
    def_l = torch.tensor(def_l,  dtype=torch.long)
    prod_l= torch.tensor(prod_l, dtype=torch.long)
    known = (def_l!=-1).nonzero(as_tuple=True)[0]
    if len(known)>=4 and torch.rand(1).item()<0.5:
        lam  = float(np.random.beta(1.0,1.0))
        lam  = max(0.3,min(0.7,lam))
        perm = known[torch.randperm(len(known))]
        B,C,H,W = imgs[known].shape
        cut_ratio=np.sqrt(1-lam); cut_h=int(H*cut_ratio)
        cut_w=int(W*cut_ratio)
        cx=np.random.randint(W); cy=np.random.randint(H)
        x1=max(0,cx-cut_w//2); x2=min(W,cx+cut_w//2)
        y1=max(0,cy-cut_h//2); y2=min(H,cy+cut_h//2)
        imgs_copy=imgs.clone()
        imgs_copy[known,:,y1:y2,x1:x2]=imgs[perm,:,y1:y2,x1:x2]
        lam_actual=1-(x2-x1)*(y2-y1)/(H*W)
        imgs=imgs_copy
        if lam_actual<0.5:
            def_l_new=def_l.clone()
            def_l_new[known]=def_l[perm]
            def_l=def_l_new
    return imgs,bin_l,def_l,prod_l,list(paths)

# ── Dataloaders ───────────────────────────────────────────────────────
sampler = make_weighted_sampler(train_df)
train_3head_loader = DataLoader(
    train_3head_ds, batch_size=32, sampler=sampler,
    num_workers=4, pin_memory=True, drop_last=True,
    collate_fn=cutmix_collate)
val_3head_loader = DataLoader(
    val_3head_ds, batch_size=32, shuffle=False,
    num_workers=4, pin_memory=True)
test_3head_loader = DataLoader(
    test_3head_ds, batch_size=32, shuffle=False,
    num_workers=4, pin_memory=True)

# ── Load main model ───────────────────────────────────────────────────
model = EdgeNetV2(NUM_DEFECT_TYPES, NUM_PRODUCT_TYPES).to(device)

# ── Load teacher ──────────────────────────────────────────────────────
teacher = EfficientNetTeacher(NUM_DEFECT_TYPES, NUM_PRODUCT_TYPES).to(device)
ck_t = torch.load(
    '/home/sufi/training_results/baseline_models/efficientnet_b0_best.pth',
    map_location=device, weights_only=False)
teacher.load_state_dict(ck_t, strict=False)
teacher.eval()
for p in teacher.parameters():
    p.requires_grad = False

# ── criterion_dict ────────────────────────────────────────────────────
class FocalLoss(nn.Module):
    def __init__(self, gamma_per_class, weight=None, ignore_index=-1):
        super().__init__()
        self.gamma        = torch.tensor(gamma_per_class,
                                         dtype=torch.float32)
        self.weight       = weight
        self.ignore_index = ignore_index

    def forward(self, logits, targets):
        valid = targets != self.ignore_index
        if valid.sum()==0:
            return logits.sum()*0.0
        logits  = logits[valid]; targets = targets[valid]
        gamma   = self.gamma.to(logits.device)
        ce      = F.cross_entropy(logits, targets,
                                  weight=self.weight, reduction='none')
        pt      = torch.exp(-ce); g = gamma[targets]
        return ((1-pt)**g * ce).mean()

class HomoscedasticLoss(nn.Module):
    def __init__(self, n_tasks=3):
        super().__init__()
        self.log_vars = nn.Parameter(torch.zeros(n_tasks))
    def weights(self):
        return torch.exp(-self.log_vars).detach()
    def set_epoch(self, epoch): pass
    def forward(self, losses):
        w = torch.exp(-self.log_vars)
        return sum(w[i]*losses[i]+self.log_vars[i]
                   for i in range(len(losses)))

def compute_defect_weights(df, n, dev):
    counts = torch.zeros(n)
    for i in range(n):
        counts[i] = max(1,(df['defect_type_label']==i).sum())
    w = 1.0/counts
    return (w/w.sum()*n).to(dev)

defect_weights = compute_defect_weights(train_df, NUM_DEFECT_TYPES, device)
GAMMA_PER_CLASS = [2.0,2.0,3.5,2.0,2.0,2.0,4.0,2.0]
n_def  = (train_df['binary_label']==1).sum()
n_norm = (train_df['binary_label']==0).sum()
pos_weight = torch.tensor([n_norm/max(n_def,1)],
                           dtype=torch.float32).to(device)
criterion_dict = {
    'binary':    nn.BCEWithLogitsLoss(pos_weight=pos_weight),
    'defect':    FocalLoss(GAMMA_PER_CLASS, defect_weights, -1),
    'product':   nn.CrossEntropyLoss(),
    'multitask': HomoscedasticLoss(3).to(device),
}

# ── Test eval helper (needed by PUSH cells) ───────────────────────────
def test_eval_single(mdl, loader, name):
    mdl.eval()
    all_pred,all_true=[],[]
    bin_c=prod_c=tot=def_c=def_tot=0
    with torch.no_grad():
        for batch in loader:
            imgs,bin_lbl,def_lbl,prod_lbl,_ = batch
            imgs=imgs.to(device); bin_lbl=bin_lbl.to(device)
            def_lbl=def_lbl.to(device); prod_lbl=prod_lbl.to(device)
            sb,sd,sp = mdl(imgs)
            bin_c  +=((torch.sigmoid(sb.squeeze())>0.5).long()
                      ==bin_lbl).sum().item()
            prod_c +=(sp.argmax(1)==prod_lbl).sum().item()
            tot    +=imgs.size(0)
            known   = def_lbl!=-1
            if known.sum()>0:
                preds    = sd[known].argmax(1)
                def_c   +=(preds==def_lbl[known]).sum().item()
                def_tot +=known.sum().item()
                all_pred.extend(preds.cpu().tolist())
                all_true.extend(def_lbl[known].cpu().tolist())
    b=100.*bin_c/max(tot,1); d=100.*def_c/max(def_tot,1)
    p2=100.*prod_c/max(tot,1)
    p,r,f,s = precision_recall_fscore_support(
        all_true, all_pred,
        labels=list(range(NUM_DEFECT_TYPES)), zero_division=0)
    return dict(name=name, bin=b, defect=d, prod=p2,
                f1=float(f.mean()), p=p, r=r, f=f, s=s)

# ── HR256 datasets (needed by PUSH_CELL_1) ───────────────────────────
hr_val_transform = transforms.Compose([
    transforms.Resize((256,256)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
hr_test_ds = ThreeHeadDataset(test_df, transform=hr_val_transform)

print(f"\n{'='*55}")
print(f"✅ SETUP_CELL COMPLETE")
print(f"   Device            : {device}")
print(f"   NUM_DEFECT_TYPES  : {NUM_DEFECT_TYPES}")
print(f"   NUM_PRODUCT_TYPES : {NUM_PRODUCT_TYPES}")
print(f"   Train             : {len(train_3head_ds):,}")
print(f"   Val               : {len(val_3head_ds):,}")
print(f"   Test              : {len(test_3head_ds):,}")
print(f"   EdgeNetV2 class   : ✅")
print(f"   Teacher           : ✅ frozen")
print(f"   criterion_dict    : ✅")
print(f"   kd_loss           : ✅")
print(f"{'='*55}")

Device: cuda

✅ SETUP_CELL COMPLETE
   Device            : cuda
   NUM_DEFECT_TYPES  : 8
   NUM_PRODUCT_TYPES : 17
   Train             : 16,337
   Val               : 3,339
   Test              : 1,668
   EdgeNetV2 class   : ✅
   Teacher           : ✅ frozen
   criterion_dict    : ✅
   kd_loss           : ✅


In [3]:
# ======================================================================
# PUSH_CELL_1 — MULTI-SCALE ENSEMBLE  (zero training)
# ENH (224×224) + HR256 (256×256) — different resolutions see
# different spatial features. Combine for best of both.
# ======================================================================
import torch
import numpy as np
from sklearn.metrics import precision_recall_fscore_support
from torch.utils.data import DataLoader

# ── Load both models fresh ────────────────────────────────────────────
model_enh_ms = EdgeNetV2(NUM_DEFECT_TYPES, NUM_PRODUCT_TYPES).to(device)
model_enh_ms.load_state_dict(
    torch.load('/home/sufi/training_results/models/EdgeNet_V2_ENH.pth',
               map_location=device, weights_only=False)['model_state_dict'])
model_enh_ms.eval()

model_hr_ms = EdgeNetV2(NUM_DEFECT_TYPES, NUM_PRODUCT_TYPES).to(device)
model_hr_ms.load_state_dict(
    torch.load('/home/sufi/training_results/models/HR256/EdgeNet_V2_ENH_HR256.pth',
               map_location=device, weights_only=False)['model_state_dict'])
model_hr_ms.eval()

model_kd_ms = EdgeNetV2(NUM_DEFECT_TYPES, NUM_PRODUCT_TYPES).to(device)
model_kd_ms.load_state_dict(
    torch.load('/home/sufi/training_results/models/EdgeNet_V2_KD.pth',
               map_location=device, weights_only=False)['model_state_dict'])
model_kd_ms.eval()

print("✅ All models loaded")

# ── TTA augmentations ─────────────────────────────────────────────────
TTA_AUGS = [
    lambda x: x,
    lambda x: torch.flip(x, [3]),
    lambda x: torch.flip(x, [2]),
    lambda x: torch.rot90(x, 1, [2, 3]),
    lambda x: torch.rot90(x, 3, [2, 3]),
]

# ── Multi-scale evaluator ─────────────────────────────────────────────
import torchvision.transforms.functional as TF

def resize_batch(imgs, size):
    """Resize a batch tensor [B,C,H,W] to (size×size)."""
    return torch.nn.functional.interpolate(
        imgs, size=(size, size),
        mode='bilinear', align_corners=False)

def multiscale_ensemble_eval(models_sizes_weights,
                              loader, use_tta=False, label=""):
    """
    models_sizes_weights: list of (model, input_size, weight)
      - model      : nn.Module (eval mode)
      - input_size : int — resize batch to this before forward
                     (None = use loader's native size)
      - weight     : float — ensemble weight
    """
    for m, _, _ in models_sizes_weights:
        m.eval()

    all_pred, all_true = [], []
    bin_c = prod_c = tot = def_c = def_tot = 0

    with torch.no_grad():
        for batch in loader:
            imgs, bin_lbl, def_lbl, prod_lbl, _ = batch
            imgs     = imgs.to(device)
            bin_lbl  = bin_lbl.to(device)
            def_lbl  = def_lbl.to(device)
            prod_lbl = prod_lbl.to(device)

            B = imgs.size(0)
            avg_bin  = torch.zeros(B,                   device=device)
            avg_def  = torch.zeros(B, NUM_DEFECT_TYPES, device=device)
            avg_prod = torch.zeros(B, NUM_PRODUCT_TYPES,device=device)

            for mdl, sz, w in models_sizes_weights:
                # Resize if needed
                inp = resize_batch(imgs, sz) if sz is not None \
                      else imgs

                augs = TTA_AUGS if use_tta else [lambda x: x]
                b_list, d_list, p_list = [], [], []

                for aug in augs:
                    sb, sd, sp = mdl(aug(inp))
                    b_list.append(torch.sigmoid(sb.squeeze(1)))
                    d_list.append(torch.softmax(sd, dim=1))
                    p_list.append(torch.softmax(sp, dim=1))

                avg_bin  += w * torch.stack(b_list).mean(0)
                avg_def  += w * torch.stack(d_list).mean(0)
                avg_prod += w * torch.stack(p_list).mean(0)

            bin_c  += ((avg_bin > 0.5).long() == bin_lbl).sum().item()
            prod_c += (avg_prod.argmax(1) == prod_lbl).sum().item()
            tot    += B
            known   = def_lbl != -1
            if known.sum() > 0:
                preds    = avg_def[known].argmax(1)
                def_c   += (preds == def_lbl[known]).sum().item()
                def_tot += known.sum().item()
                all_pred.extend(preds.cpu().tolist())
                all_true.extend(def_lbl[known].cpu().tolist())

    bin_acc  = 100. * bin_c  / max(tot,     1)
    def_acc  = 100. * def_c  / max(def_tot, 1)
    prod_acc = 100. * prod_c / max(tot,     1)
    p, r, f, s = precision_recall_fscore_support(
        all_true, all_pred,
        labels=list(range(NUM_DEFECT_TYPES)), zero_division=0)
    f1 = float(f.mean())

    print(f"  {label:<50} Def={def_acc:.2f}%  "
          f"F1={f1:.3f}  Bin={bin_acc:.2f}%")
    return dict(label=label, bin=bin_acc, defect=def_acc,
                prod=prod_acc, f1=f1, p=p, r=r, f=f, s=s)


# ── Use the 224-normalised test loader for all (resize inside) ────────
test_loader_ms = DataLoader(
    test_3head_ds, batch_size=32, shuffle=False,
    num_workers=4, pin_memory=True)

print("\n" + "="*70)
print("MULTI-SCALE ENSEMBLE RESULTS  (Test Set)")
print("="*70)

# Baselines
r_enh_only = multiscale_ensemble_eval(
    [(model_enh_ms, None, 1.0)],
    test_loader_ms, use_tta=False,
    label="ENH only  (224)")

r_hr_only = multiscale_ensemble_eval(
    [(model_hr_ms, 256, 1.0)],
    test_loader_ms, use_tta=False,
    label="HR256 only  (256)")

# Multi-scale pairs
r_ms_5050 = multiscale_ensemble_eval(
    [(model_enh_ms, None, 0.5),
     (model_hr_ms,  256,  0.5)],
    test_loader_ms, use_tta=False,
    label="ENH×0.5 + HR256×0.5  (multi-scale)")

r_ms_6040 = multiscale_ensemble_eval(
    [(model_enh_ms, None, 0.6),
     (model_hr_ms,  256,  0.4)],
    test_loader_ms, use_tta=False,
    label="ENH×0.6 + HR256×0.4  (multi-scale)")

r_ms_7030 = multiscale_ensemble_eval(
    [(model_enh_ms, None, 0.7),
     (model_hr_ms,  256,  0.3)],
    test_loader_ms, use_tta=False,
    label="ENH×0.7 + HR256×0.3  (multi-scale)")

# Triple: ENH + HR256 + KD
r_triple = multiscale_ensemble_eval(
    [(model_enh_ms, None, 0.5),
     (model_hr_ms,  256,  0.3),
     (model_kd_ms,  None, 0.2)],
    test_loader_ms, use_tta=False,
    label="ENH×0.5 + HR256×0.3 + KD×0.2")

# Best combo + TTA
r_ms_tta = multiscale_ensemble_eval(
    [(model_enh_ms, None, 0.6),
     (model_hr_ms,  256,  0.4)],
    test_loader_ms, use_tta=True,
    label="ENH×0.6 + HR256×0.4 + TTA  ★")

r_triple_tta = multiscale_ensemble_eval(
    [(model_enh_ms, None, 0.5),
     (model_hr_ms,  256,  0.3),
     (model_kd_ms,  None, 0.2)],
    test_loader_ms, use_tta=True,
    label="Triple + TTA  ★★")

print("="*70)
all_r = [r_enh_only, r_hr_only, r_ms_5050, r_ms_6040,
         r_ms_7030, r_triple, r_ms_tta, r_triple_tta]
best  = max(all_r, key=lambda x: x['f1'])
print(f"\n  Best multi-scale F1 : {best['f1']:.3f}  ({best['label']})")
print("\n✅ PUSH_CELL_1 DONE")

✅ All models loaded

MULTI-SCALE ENSEMBLE RESULTS  (Test Set)
  ENH only  (224)                                    Def=89.86%  F1=0.864  Bin=98.32%
  HR256 only  (256)                                  Def=85.14%  F1=0.813  Bin=98.44%
  ENH×0.5 + HR256×0.5  (multi-scale)                 Def=87.16%  F1=0.836  Bin=98.56%
  ENH×0.6 + HR256×0.4  (multi-scale)                 Def=88.51%  F1=0.851  Bin=98.44%
  ENH×0.7 + HR256×0.3  (multi-scale)                 Def=88.51%  F1=0.851  Bin=98.38%
  ENH×0.5 + HR256×0.3 + KD×0.2                       Def=88.51%  F1=0.851  Bin=98.44%
  ENH×0.6 + HR256×0.4 + TTA  ★                       Def=87.16%  F1=0.835  Bin=98.68%
  Triple + TTA  ★★                                   Def=87.84%  F1=0.849  Bin=98.80%

  Best multi-scale F1 : 0.864  (ENH only  (224))

✅ PUSH_CELL_1 DONE


In [4]:
# ======================================================================
# PUSH_CELL_2 — COSINE RESTART  (20 epochs, RandAugment on hard classes)
# Continues from ENH, applies stronger augmentation to the 3 hard classes
# deformation / cut / scratch get extra augmentation probability
# Saves → /models/ENH_R2/EdgeNet_V2_ENH_R2.pth
# ======================================================================
import torch, numpy as np, random
from pathlib import Path
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from sklearn.metrics import precision_recall_fscore_support
from torch.cuda.amp import GradScaler
from PIL import Image
import cv2

# ── Save directory ────────────────────────────────────────────────────
R2_DIR  = Path('/home/sufi/training_results/models/ENH_R2')
R2_DIR.mkdir(parents=True, exist_ok=True)
R2_SAVE = R2_DIR / 'EdgeNet_V2_ENH_R2.pth'

R2_EPOCHS   = 20
HARD_CLASSES = {
    DEFECT_TYPE_TO_IDX['deformation'],
    DEFECT_TYPE_TO_IDX['cut'],
    DEFECT_TYPE_TO_IDX['scratch'],
}

# ── Hard-class dataset: stronger augmentation for hard classes ─────────
class HardClassDataset(Dataset):
    """
    Same as ThreeHeadDataset but applies stronger RandAugment
    to samples belonging to hard defect classes.
    """
    def __init__(self, df, normal_transform, hard_transform,
                 hard_classes):
        self.df               = df.reset_index(drop=True)
        self.normal_transform = normal_transform
        self.hard_transform   = hard_transform
        self.hard_classes     = hard_classes

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        img_path = row['image_path']
        img      = cv2.imread(img_path)
        if img is None:
            img = np.zeros((224, 224, 3), dtype=np.uint8)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = Image.fromarray(img)

        def_lbl = int(row['defect_type_label'])
        is_hard = def_lbl in self.hard_classes

        if is_hard:
            img = self.hard_transform(img)
        else:
            img = self.normal_transform(img)

        return (img,
                int(row['binary_label']),
                def_lbl,
                int(row['product_type_label']),
                img_path)


# ── Transforms ────────────────────────────────────────────────────────
normal_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(0.3, 0.3, 0.2, 0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

hard_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),            # stronger rotation
    transforms.ColorJitter(0.4, 0.4, 0.3, 0.15),
    transforms.RandomAffine(
        degrees=15,
        translate=(0.1, 0.1),
        scale=(0.85, 1.15),
        shear=10,
    ),
    transforms.RandomAutocontrast(p=0.4),
    transforms.RandomAdjustSharpness(2, p=0.4),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

# ── Dataset + loader ──────────────────────────────────────────────────
r2_train_ds = HardClassDataset(
    train_df,
    normal_transform = normal_tf,
    hard_transform   = hard_tf,
    hard_classes     = HARD_CLASSES,
)

# Oversample hard classes 3× compared to easy classes
r2_weights = []
for _, row in train_df.iterrows():
    lbl = int(row['defect_type_label'])
    if lbl in HARD_CLASSES:
        r2_weights.append(3.0)    # 3× oversampling for hard classes
    elif lbl == -1:
        r2_weights.append(0.3)    # same as original
    else:
        r2_weights.append(1.0)

r2_sampler = WeightedRandomSampler(
    torch.tensor(r2_weights, dtype=torch.float32),
    len(r2_weights), replacement=True)

r2_train_loader = DataLoader(
    r2_train_ds,
    batch_size  = 32,
    sampler     = r2_sampler,
    num_workers = 4,
    pin_memory  = True,
    drop_last   = True,
    collate_fn  = cutmix_collate,
)

print(f"✅ R2 dataset ready")
print(f"   Train batches : {len(r2_train_loader)}")
print(f"   Hard classes  : deformation, cut, scratch (3× oversampled)")

# ── Load ENH checkpoint ───────────────────────────────────────────────
ck_enh = torch.load(
    '/home/sufi/training_results/models/EdgeNet_V2_ENH.pth',
    map_location=device, weights_only=False)
model.load_state_dict(ck_enh['model_state_dict'])
model.freeze_backbone(False)
print(f"Loaded ENH epoch {ck_enh['epoch']}  "
      f"val F1={ck_enh['val_metrics']['defect_f1_macro']:.3f}")

# ── Optimizer — warmup restart ────────────────────────────────────────
bb_params    = list(model.early_features.parameters()) + \
               list(model.late_features.parameters())
coord_params = list(model.coord_att1.parameters()) + \
               list(model.coord_att2.parameters())
head_params  = list(model.shared.parameters())      + \
               list(model.binary_head.parameters()) + \
               list(model.defect_head.parameters()) + \
               list(model.product_head.parameters())

optimizer_r2 = torch.optim.AdamW([
    {'params': bb_params,    'lr': 5e-6},
    {'params': coord_params, 'lr': 2e-5},
    {'params': head_params,  'lr': 2e-5},
], weight_decay=1e-4)

scheduler_r2 = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer_r2,
    T_0    = 10,      # restart every 10 epochs
    T_mult = 1,
    eta_min= 1e-7,
)
scaler_r2 = GradScaler()

# ── Training ───────────────────────────────────────────────────────────
best_f1_r2 = 0.0

def evaluate_r2(model, loader):
    model.eval()
    all_pred, all_true = [], []
    bin_c = prod_c = tot = def_c = def_tot = 0
    with torch.no_grad():
        for batch in loader:
            imgs, bin_lbl, def_lbl, prod_lbl, _ = batch
            imgs     = imgs.to(device)
            bin_lbl  = bin_lbl.to(device)
            def_lbl  = def_lbl.to(device)
            prod_lbl = prod_lbl.to(device)
            sb, sd, sp = model(imgs)
            bin_c  += ((torch.sigmoid(sb.squeeze()) > 0.5).long()
                       == bin_lbl).sum().item()
            prod_c += (sp.argmax(1) == prod_lbl).sum().item()
            tot    += imgs.size(0)
            known   = def_lbl != -1
            if known.sum() > 0:
                preds    = sd[known].argmax(1)
                def_c   += (preds == def_lbl[known]).sum().item()
                def_tot += known.sum().item()
                all_pred.extend(preds.cpu().tolist())
                all_true.extend(def_lbl[known].cpu().tolist())
    bin_acc  = 100.*bin_c/max(tot,1)
    def_acc  = 100.*def_c/max(def_tot,1)
    prod_acc = 100.*prod_c/max(tot,1)
    p, r, f, _ = precision_recall_fscore_support(
        all_true, all_pred,
        labels=list(range(NUM_DEFECT_TYPES)), zero_division=0)
    return bin_acc, def_acc, prod_acc, float(f.mean()), p, r, f

for epoch in range(1, R2_EPOCHS + 1):

    model.train()
    criterion_dict['multitask'].set_epoch(epoch)
    kd_loss.set_alpha(0.5)

    tr_bin_c = tr_def_c = tr_prod_c = tr_def_tot = tr_tot = 0
    tr_pred, tr_true = [], []

    for batch in r2_train_loader:
        imgs, bin_lbl, def_lbl, prod_lbl, _ = batch
        imgs     = imgs.to(device)
        bin_lbl  = bin_lbl.to(device)
        def_lbl  = def_lbl.to(device)
        prod_lbl = prod_lbl.to(device)

        optimizer_r2.zero_grad()

        with torch.cuda.amp.autocast():
            s_bin, s_def, s_prod = model(imgs)
            with torch.no_grad():
                t_bin, t_def, t_prod = teacher(imgs)

            s_bin_sq = s_bin.squeeze()
            t_bin_sq = t_bin.squeeze()
            known    = def_lbl != -1

            hard_bin  = criterion_dict['binary'](s_bin_sq, bin_lbl.float())
            hard_prod = criterion_dict['product'](s_prod, prod_lbl)

            if known.sum() > 0:
                hard_def = criterion_dict['defect'](
                    s_def[known], def_lbl[known])
                soft_def = kd_loss._kl(
                    s_def[known], t_def[known], kd_loss.T_def)
            else:
                hard_def = torch.tensor(0.0, device=device)
                soft_def = torch.tensor(0.0, device=device)

            s_b2 = torch.stack([-s_bin_sq, s_bin_sq], dim=1)
            t_b2 = torch.stack([-t_bin_sq, t_bin_sq], dim=1)
            soft_bin  = kd_loss._kl(s_b2,   t_b2,   kd_loss.T_bin)
            soft_prod = kd_loss._kl(s_prod, t_prod, kd_loss.T_prod)

            a  = kd_loss.alpha
            lb = a * hard_bin  + (1 - a) * soft_bin
            ld = a * hard_def  + (1 - a) * soft_def
            lp = a * hard_prod + (1 - a) * soft_prod

            tw         = criterion_dict['multitask'].weights()
            total_loss = (float(tw[0])*lb +
                          float(tw[1])*ld +
                          float(tw[2])*lp)

        scaler_r2.scale(total_loss).backward()
        scaler_r2.unscale_(optimizer_r2)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler_r2.step(optimizer_r2)
        scaler_r2.update()

        tr_tot    += imgs.size(0)
        tr_bin_c  += ((torch.sigmoid(s_bin_sq.detach()) > 0.5).long()
                      == bin_lbl).sum().item()
        tr_prod_c += (s_prod.detach().argmax(1) == prod_lbl).sum().item()
        if known.sum() > 0:
            preds = s_def.detach()[known].argmax(1)
            tr_def_c   += (preds == def_lbl[known]).sum().item()
            tr_def_tot += known.sum().item()
            tr_pred.extend(preds.cpu().tolist())
            tr_true.extend(def_lbl[known].cpu().tolist())

    scheduler_r2.step()

    t_def_acc = 100.*tr_def_c/max(tr_def_tot,1)
    if tr_true:
        _, _, tf, _ = precision_recall_fscore_support(
            tr_true, tr_pred,
            labels=list(range(NUM_DEFECT_TYPES)), zero_division=0)
        t_f1 = float(tf.mean())
    else:
        t_f1 = 0.0

    v_bin, v_def, v_prod, v_f1, vp, vr, vf = evaluate_r2(
        model, val_3head_loader)
    cur_lr = optimizer_r2.param_groups[1]['lr']
    tw     = criterion_dict['multitask'].weights()

    print(f"\nEpoch [{epoch}/{R2_EPOCHS}]  LR={cur_lr:.1e}  "
          f"TaskW=[bin={float(tw[0]):.2f} def={float(tw[1]):.2f} "
          f"prod={float(tw[2]):.2f}]  [ENH_R2+HardAug]")
    print()
    print(f"  Train → Def={t_def_acc:.1f}%  F1={t_f1:.3f}")
    print(f"  Val   → Bin={v_bin:.1f}%  Def={v_def:.1f}%  "
          f"F1={v_f1:.3f}  Prod={v_prod:.1f}%")
    print()
    print(f"  {'Class':<22} {'P':>6}  {'R':>6}  {'F1':>6}")
    print(f"  {'-'*40}")
    for i, name in enumerate(SEMANTIC_GROUPS):
        star = " ←hard" if DEFECT_TYPE_TO_IDX.get(name, -1) \
               in HARD_CLASSES else ""
        print(f"  {name:<22} {vp[i]:>6.3f}  {vr[i]:>6.3f}  "
              f"{vf[i]:>6.3f}{star}")
    print(f"  {'-'*40}")
    print(f"  {'MACRO':<22} {float(np.mean(vp)):>6.3f}  "
          f"{float(np.mean(vr)):>6.3f}  {v_f1:>6.3f}")

    if v_f1 > best_f1_r2:
        best_f1_r2 = v_f1
        torch.save({
            'epoch':               epoch,
            'model_state_dict':    model.state_dict(),
            'multitask_state':     criterion_dict['multitask'].state_dict()
                                   if hasattr(criterion_dict['multitask'],
                                              'state_dict') else None,
            'optimizer_state':     optimizer_r2.state_dict(),
            'val_score':           v_f1,
            'val_metrics': {
                'binary_acc':      v_bin,
                'defect_acc':      v_def,
                'defect_f1_macro': v_f1,
                'product_acc':     v_prod,
            },
            'parent_checkpoint':   'EdgeNet_V2_ENH.pth',
            'hard_classes':        list(HARD_CLASSES),
            'defect_type_to_idx':  DEFECT_TYPE_TO_IDX,
            'product_type_to_idx': PRODUCT_TYPE_TO_IDX,
            'idx_to_defect_type':  IDX_TO_DEFECT_TYPE,
            'idx_to_product_type': IDX_TO_PRODUCT_TYPE,
        }, R2_SAVE)
        print(f"\n  ✅ NEW BEST  F1={best_f1_r2:.3f} "
              f"→ saved ENH_R2/EdgeNet_V2_ENH_R2.pth")

print(f"\n{'='*60}")
print(f"ENH_R2 TRAINING COMPLETE")
print(f"  Best Val F1  : {best_f1_r2:.3f}")
print(f"  Saved to     : {R2_SAVE}")
print(f"{'='*60}")
print("✅ PUSH_CELL_2 DONE")

✅ R2 dataset ready
   Train batches : 510
   Hard classes  : deformation, cut, scratch (3× oversampled)
Loaded ENH epoch 23  val F1=0.896


/tmp/ipykernel_1159/666771562.py:168: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler_r2 = GradScaler()
/tmp/ipykernel_1159/666771562.py:222: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():



Epoch [1/20]  LR=2.0e-05  TaskW=[bin=1.00 def=1.00 prod=1.00]  [ENH_R2+HardAug]

  Train → Def=73.9%  F1=0.715
  Val   → Bin=97.8%  Def=86.5%  F1=0.832  Prod=99.5%

  Class                       P       R      F1
  ----------------------------------------
  contamination           0.842   0.970   0.901
  cut                     0.833   0.882   0.857 ←hard
  deformation             0.800   0.533   0.640 ←hard
  fracture                0.826   0.884   0.854
  hole_void               0.944   0.944   0.944
  minor_defect            0.780   0.929   0.848
  scratch                 0.895   0.586   0.708 ←hard
  surface_quality         0.918   0.889   0.903
  ----------------------------------------
  MACRO                   0.855   0.827   0.832

  ✅ NEW BEST  F1=0.832 → saved ENH_R2/EdgeNet_V2_ENH_R2.pth


/tmp/ipykernel_1159/666771562.py:222: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():



Epoch [2/20]  LR=1.8e-05  TaskW=[bin=1.00 def=1.00 prod=1.00]  [ENH_R2+HardAug]

  Train → Def=75.1%  F1=0.734
  Val   → Bin=98.1%  Def=86.5%  F1=0.833  Prod=99.4%

  Class                       P       R      F1
  ----------------------------------------
  contamination           0.842   0.970   0.901
  cut                     0.882   0.882   0.882 ←hard
  deformation             0.727   0.533   0.615 ←hard
  fracture                0.851   0.930   0.889
  hole_void               0.962   0.944   0.953
  minor_defect            0.760   0.905   0.826
  scratch                 0.895   0.586   0.708 ←hard
  surface_quality         0.902   0.873   0.887
  ----------------------------------------
  MACRO                   0.853   0.828   0.833

  ✅ NEW BEST  F1=0.833 → saved ENH_R2/EdgeNet_V2_ENH_R2.pth


...


Epoch [18/20]  LR=2.0e-06  TaskW=[bin=1.00 def=1.00 prod=1.00]  [ENH_R2+HardAug]

  Train → Def=77.5%  F1=0.762
  Val   → Bin=98.1%  Def=86.1%  F1=0.821  Prod=99.7%

  Class                       P       R      F1
  ----------------------------------------
  contamination           0.838   0.939   0.886
  cut                     0.923   0.706   0.800 ←hard
  deformation             0.800   0.533   0.640 ←hard
  fracture                0.774   0.953   0.854
  hole_void               0.962   0.944   0.953
  minor_defect            0.759   0.976   0.854
  scratch                 0.938   0.517   0.667 ←hard
  surface_quality         0.933   0.889   0.911
  ----------------------------------------
  MACRO                   0.866   0.807   0.821


/tmp/ipykernel_1159/666771562.py:222: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():



Epoch [19/20]  LR=5.9e-07  TaskW=[bin=1.00 def=1.00 prod=1.00]  [ENH_R2+HardAug]

  Train → Def=77.1%  F1=0.758
  Val   → Bin=98.2%  Def=87.2%  F1=0.839  Prod=99.7%

  Class                       P       R      F1
  ----------------------------------------
  contamination           0.838   0.939   0.886
  cut                     0.933   0.824   0.875 ←hard
  deformation             0.889   0.533   0.667 ←hard
  fracture                0.774   0.953   0.854
  hole_void               0.962   0.944   0.953
  minor_defect            0.788   0.976   0.872
  scratch                 0.941   0.552   0.696 ←hard
  surface_quality         0.933   0.889   0.911
  ----------------------------------------
  MACRO                   0.882   0.826   0.839


/tmp/ipykernel_1159/666771562.py:222: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():



Epoch [20/20]  LR=2.0e-05  TaskW=[bin=1.00 def=1.00 prod=1.00]  [ENH_R2+HardAug]

  Train → Def=77.3%  F1=0.761
  Val   → Bin=98.2%  Def=87.2%  F1=0.837  Prod=99.7%

  Class                       P       R      F1
  ----------------------------------------
  contamination           0.838   0.939   0.886
  cut                     0.933   0.824   0.875 ←hard
  deformation             0.889   0.533   0.667 ←hard
  fracture                0.778   0.977   0.866
  hole_void               0.981   0.944   0.962
  minor_defect            0.759   0.976   0.854
  scratch                 0.938   0.517   0.667 ←hard
  surface_quality         0.949   0.889   0.918
  ----------------------------------------
  MACRO                   0.883   0.825   0.837

ENH_R2 TRAINING COMPLETE
  Best Val F1  : 0.852
  Saved to     : /home/sufi/training_results/models/ENH_R2/EdgeNet_V2_ENH_R2.pth
✅ PUSH_CELL_2 DONE


In [ ]:
# ======================================================================
# PUSH_CELL_3 — ULTIMATE FINAL TABLE
# All models × all boost strategies on test set
# ======================================================================
from sklearn.metrics import precision_recall_fscore_support
import numpy as np

test_loader_final = DataLoader(
    test_3head_ds, batch_size=32, shuffle=False,
    num_workers=4, pin_memory=True)

print("Loading ENH_R2...")
model_r2 = EdgeNetV2(NUM_DEFECT_TYPES, NUM_PRODUCT_TYPES).to(device)
model_r2.load_state_dict(
    torch.load('/home/sufi/training_results/models/ENH_R2/'
               'EdgeNet_V2_ENH_R2.pth',
               map_location=device, weights_only=False)['model_state_dict'])
model_r2.eval()

# Reload ENH and HR256 fresh
model_enh_f = EdgeNetV2(NUM_DEFECT_TYPES, NUM_PRODUCT_TYPES).to(device)
model_enh_f.load_state_dict(
    torch.load('/home/sufi/training_results/models/EdgeNet_V2_ENH.pth',
               map_location=device, weights_only=False)['model_state_dict'])
model_enh_f.eval()

model_hr_f = EdgeNetV2(NUM_DEFECT_TYPES, NUM_PRODUCT_TYPES).to(device)
model_hr_f.load_state_dict(
    torch.load('/home/sufi/training_results/models/HR256/'
               'EdgeNet_V2_ENH_HR256.pth',
               map_location=device, weights_only=False)['model_state_dict'])
model_hr_f.eval()
print("✅ All models loaded")

print("\nRunning all evaluations...\n")

# ── Individual models ─────────────────────────────────────────────────
r_orig  = test_eval_single(model_orig,  test_loader_final, "EdgeNet-V2")
r_kd    = test_eval_single(model_kd_ms, test_loader_final, "EdgeNet-V2+KD")
r_enh   = test_eval_single(model_enh_f, test_loader_final, "EdgeNet-V2+ENH")
r_r2    = test_eval_single(model_r2,    test_loader_final, "EdgeNet-V2+R2")
r_tea   = test_eval_single(teacher,     test_loader_final,
                            "EfficientNet-B0 (Teacher)")

# ── Zero-cost boosts on R2 ────────────────────────────────────────────
r_r2_tta = multiscale_ensemble_eval(
    [(model_r2, None, 1.0)],
    test_loader_final, use_tta=True,
    label="EdgeNet-V2+R2+TTA")

# ── Best multi-scale combinations with R2 ────────────────────────────
r_r2_enh = multiscale_ensemble_eval(
    [(model_r2, None, 0.6), (model_enh_f, None, 0.4)],
    test_loader_final, use_tta=False,
    label="R2×0.6 + ENH×0.4")

r_r2_enh_tta = multiscale_ensemble_eval(
    [(model_r2, None, 0.6), (model_enh_f, None, 0.4)],
    test_loader_final, use_tta=True,
    label="R2×0.6 + ENH×0.4 + TTA")

r_triple_tta = multiscale_ensemble_eval(
    [(model_r2,    None, 0.5),
     (model_enh_f, None, 0.3),
     (model_hr_f,  256,  0.2)],
    test_loader_final, use_tta=True,
    label="R2×0.5 + ENH×0.3 + HR256×0.2 + TTA  ★★")

# ══════════════════════════════════════════════════════════════════════
# ULTIMATE TABLE
# ══════════════════════════════════════════════════════════════════════
print()
print("=" * 78)
print("ULTIMATE THESIS RESULTS — TEST SET")
print("=" * 78)
hdr = (f"  {'Model':<40} {'Params':>7}  "
       f"{'Binary':>8}  {'Defect':>8}  {'F1':>7}  {'Product':>9}")
div = "  " + "-" * 74
print(hdr); print(div)

for name, defect in [("MobileNetV3-Small", "79.05%"),
                      ("ResNet-18",         "84.46%"),
                      ("ResNet-50",         "90.88%"),
                      ("MobileNetV3-Large", "91.22%")]:
    print(f"  {name:<40} {'—':>7}  {'—':>8}  "
          f"{defect:>8}  {'—':>7}  {'—':>9}")
print(div)
print(f"  {r_tea['name']:<40} {'5.00M':>7}  "
      f"{r_tea['bin']:>7.2f}%  {r_tea['defect']:>7.2f}%  "
      f"{r_tea['f1']:>7.3f}  {r_tea['prod']:>8.2f}%  ← Teacher")
print(div)

all_results = [
    (r_orig,       "3.89M", ""),
    (r_kd,         "3.89M", "  +KD"),
    (r_enh,        "3.89M", "  +Feat+SupCon"),
    (r_r2,         "3.89M", "  +HardAug+Restart"),
    (r_r2_tta,     "3.89M", "  +TTA"),
    (r_r2_enh,     "3.89M", "  +Ensemble"),
    (r_r2_enh_tta, "3.89M", "  +Ensemble+TTA"),
    (r_triple_tta, "3.89M", "  ★ BEST"),
]
best_f1 = max(r['f1'] for r, _, _ in all_results)

for res, params, tag in all_results:
    name = res.get('name', res.get('label', ''))
    star = "  ← BEST" if abs(res['f1'] - best_f1) < 1e-6 else ""
    print(f"  {name:<40} {params:>7}  "
          f"{res['bin']:>7.2f}%  {res['defect']:>7.2f}%  "
          f"{res['f1']:>7.3f}  {res['prod']:>8.2f}%{star}")
print("=" * 78)

# ── Per-class final progression ───────────────────────────────────────
best_res = max(all_results, key=lambda x: x[0]['f1'])[0]
print()
print("=" * 82)
print("PER-CLASS F1 — FULL PROGRESSION  (Test Set)")
print("=" * 82)
print(f"  {'Class':<22}  {'Base':>7}  {'KD':>7}  {'ENH':>7}  "
      f"{'R2':>7}  {'BEST':>9}  {'Total↑':>8}  {'Sup':>5}")
print("  " + "-" * 76)
for i, name in enumerate(SEMANTIC_GROUPS):
    f0  = r_orig['f'][i]
    fk  = r_kd['f'][i]
    fe  = r_enh['f'][i]
    fr2 = r_r2['f'][i]
    fb  = best_res['f'][i]
    g   = fb - f0
    flag = " ▲" if g > 0.05 else (" ▼" if g < -0.02 else "")
    print(f"  {name:<22}  {f0:>7.3f}  {fk:>7.3f}  {fe:>7.3f}  "
          f"{fr2:>7.3f}  {fb:>9.3f}  {g:>+8.3f}  "
          f"{int(best_res['s'][i]):>5}{flag}")
print("  " + "-" * 76)
print(f"  {'MACRO':<22}  {r_orig['f1']:>7.3f}  {r_kd['f1']:>7.3f}  "
      f"{r_enh['f1']:>7.3f}  {r_r2['f1']:>7.3f}  "
      f"{best_res['f1']:>9.3f}  "
      f"{best_res['f1']-r_orig['f1']:>+8.3f}")
print("=" * 82)

# ── Teacher gap summary ───────────────────────────────────────────────
print()
print("=" * 60)
print("TEACHER GAP ANALYSIS")
print("=" * 60)
gap = r_tea['defect'] - r_orig['defect']
for res, label in [
    (r_kd,         "+KD"),
    (r_enh,        "+ENH"),
    (r_r2,         "+R2"),
    (best_res,     "★ BEST (ensemble)"),
]:
    d   = res['defect'] - r_orig['defect']
    pct = 100. * d / max(gap, 1e-6)
    print(f"  {label:<28} +{d:.2f}%  "
          f"({pct:.1f}% of teacher gap closed)")
print(f"\n  Param savings vs teacher : 22.2%  (3.89M vs 5.00M)")
print("=" * 60)
print("\n✅ PUSH_CELL_3 DONE — Ultimate thesis evaluation complete")